In [1]:
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import jax
print(jax.default_backend())
print(jax.devices())
print(jax.device_count())

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1773742346.493251      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


tpu
[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]
8


In [3]:
import subprocess
import sys
import io
import signal
import os
import re
import gc
import math
import json
import time
import warnings
import pickle
from datetime import datetime
from functools import partial
from typing import Any, Tuple, Optional

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import random as py_random
from jax import random, grad, jit, vmap, tree_util
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PartitionSpec as P, NamedSharding

import optax
from flax import linen as nn
from flax.training import train_state

from sklearn.metrics import roc_auc_score

import tiktoken
from tabulate import tabulate
from tqdm import tqdm
from IPython.display import HTML, Image, Markdown

warnings.filterwarnings('ignore')

In [4]:
devices = jax.devices()
print(f"Found {len(devices)} devices: {devices}")
from flax.jax_utils import replicate, unreplicate
num_devices = len(devices)
mesh = Mesh(mesh_utils.create_device_mesh((num_devices,)), axis_names=('data',))
print(f"Mesh: {mesh}")


pd.set_option('display.max_rows', None)
pd.set_option('display.min_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth',None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)



!rm -rf /kaggle/working/*

Found 8 devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]
Mesh: Mesh('data': 8, axis_types=(Auto,))


# Step 1: Pretrain the Base Model

In [5]:
df=pd.read_csv("/kaggle/input/mini-llm-code/code_dataset.csv",nrows=10000)
print("#"*180)
print(f"Data Shape: {df.shape}")
print(f"Check Null values:\n{df.isnull().sum()}")
print("#"*180)
df=df.dropna()
df.drop(columns=["lang","stars"],inplace=True)
display(df.head())

####################################################################################################################################################################################
Data Shape: (10000, 3)
Check Null values:
text     0
lang     0
stars    0
dtype: int64
####################################################################################################################################################################################


text
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [6]:
def remove_garbage(df, text_col="text", min_chars=20):
    df = df.copy()
    
    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = re.sub(r"<\/?documents?>", "", text, flags=re.IGNORECASE)
        text = re.sub(r"<\/?document[^>]*>", "", text, flags=re.IGNORECASE)
        text = re.sub(r"<\/?document_content[^>]*>", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\badd\s+Code\b", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\badd\s+Markdown\b", "", text, flags=re.IGNORECASE)
        text = re.sub(r"<[^>]+>", "", text)
        lines = [line.rstrip() for line in text.splitlines() if line.strip()]
        return "\n".join(lines)
    
    cleaned_texts = []
    for text in tqdm(df[text_col], desc="Cleaning text", total=len(df)):
        cleaned_texts.append(clean_text(text))
    
    df[text_col] = cleaned_texts
    df = df[df[text_col].str.len() >= min_chars]
    return df.reset_index(drop=True)

In [7]:
df_clean = remove_garbage(df, text_col="text")
print(f"Cleaned dataset has {len(df_clean)} rows")

Cleaning text: 100%|██████████| 10000/10000 [00:01<00:00, 5019.83it/s]

Cleaned dataset has 10000 rows


In [8]:
df.head()

text
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [9]:
tokenizer = tiktoken.get_encoding("gpt2")

all_tokens = []
for text in df['text'].values:
    tokens = tokenizer.encode(text, allowed_special="all")
    all_tokens.extend(tokens)

all_tokens = np.array(all_tokens, dtype=np.uint16)
print(f"Total tokens: {len(all_tokens):,}")

np.save("/kaggle/working/train_data.npy", all_tokens)

Total tokens: 33,363,447


In [10]:
VOCAB_SIZE = 50257
MAX_SEQ_LEN = 256
D_MODEL = 768
NUM_LAYERS = 6
NUM_HEADS = 12
HEAD_DIM = 64
MLP_DIM = D_MODEL * 4
DROPOUT = 0.1
BATCH_SIZE = 128
LEARNING_RATE = 6e-4
WARMUP_STEPS = 2000
MAX_STEPS = 500000
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.1


def load_data(path):
    return np.load(path,allow_pickle=True)
data_path="/kaggle/working/train_data.npy"

data = load_data(data_path)
total_tokens = len(data)  
tokens_per_step = BATCH_SIZE * MAX_SEQ_LEN 
steps_per_epoch = total_tokens // tokens_per_step
total_epochs = MAX_STEPS / steps_per_epoch

print(f"Total tokens: {total_tokens:,}")
print(f"Steps per epoch: {steps_per_epoch:,}")
print(f"500k steps = {total_epochs:.1f} epochs")

Total tokens: 33,363,447
Steps per epoch: 1,018
500k steps = 491.2 epochs


In [11]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (jnp.arange(0, dim, 2)[: (dim // 2)].astype(jnp.float32) / dim))
    t = jnp.arange(end, dtype=jnp.float32)
    freqs = jnp.outer(t, freqs)
    freqs_cis = jnp.exp(1j * freqs)
    return freqs_cis

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = xq.astype(jnp.float32).reshape(*xq.shape[:-1], -1, 2)
    xk_ = xk.astype(jnp.float32).reshape(*xk.shape[:-1], -1, 2)
    xq_complex = jax.lax.complex(xq_[..., 0], xq_[..., 1])
    xk_complex = jax.lax.complex(xk_[..., 0], xk_[..., 1])
    
    freqs_cis = freqs_cis[:xq_.shape[1]]
    
    shape = [d if i == 1 or i == xq_complex.ndim - 1 else 1 for i, d in enumerate(xq_complex.shape)]
    freqs_cis = freqs_cis.reshape(shape)
    
    xq_out = xq_complex * freqs_cis
    xk_out = xk_complex * freqs_cis
    
    xq_out = jnp.stack([jnp.real(xq_out), jnp.imag(xq_out)], axis=-1).reshape(*xq.shape)
    xk_out = jnp.stack([jnp.real(xk_out), jnp.imag(xk_out)], axis=-1).reshape(*xk.shape)
    
    return xq_out.astype(xq.dtype), xk_out.astype(xk.dtype)

In [12]:
class Attention(nn.Module):
    num_heads: int
    head_dim: int
    
    @nn.compact
    def __call__(self, x, freqs_cis, mask, train, cache=None, cache_pos=None):
        B, L, D = x.shape
        q = nn.Dense(self.num_heads * self.head_dim, use_bias=False, name='q_proj')(x)
        k = nn.Dense(self.num_heads * self.head_dim, use_bias=False, name='k_proj')(x)
        v = nn.Dense(self.num_heads * self.head_dim, use_bias=False, name='v_proj')(x)
        
        q = q.reshape(B, L, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        k = k.reshape(B, L, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        v = v.reshape(B, L, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        
        q, k = apply_rotary_emb(q, k, freqs_cis)
        
        if cache is not None and cache_pos is not None:
            k = jax.lax.dynamic_update_slice(cache['k'], k, (0, 0, cache_pos, 0))
            v = jax.lax.dynamic_update_slice(cache['v'], v, (0, 0, cache_pos, 0))
            k = k[:, :, :cache_pos + L, :]
            v = v[:, :, :cache_pos + L, :]
        
        scores = jnp.matmul(q, k.transpose(0, 1, 3, 2)) / jnp.sqrt(self.head_dim)
        
        attn_mask = mask[-L:, :k.shape[2]][None, None, :, :]
        scores = jnp.where(attn_mask, scores, -1e10)
        
        attn = jax.nn.softmax(scores, axis=-1)
        attn = nn.Dropout(0.1, deterministic=not train)(attn)
        
        out = jnp.matmul(attn, v)
        out = out.transpose(0, 2, 1, 3).reshape(B, L, -1)
        out = nn.Dense(D, use_bias=False, name='o_proj')(out)
        
        return out, {'k': k, 'v': v} if cache is not None else None

class MLP(nn.Module):
    hidden_dim: int
    d_model: int
    
    @nn.compact
    def __call__(self, x, train):
        gate = nn.Dense(self.hidden_dim, use_bias=False, name='gate_proj')(x)
        up = nn.Dense(self.hidden_dim, use_bias=False, name='up_proj')(x)
        x = nn.silu(gate) * up
        x = nn.Dropout(0.1, deterministic=not train)(x)
        x = nn.Dense(self.d_model, use_bias=False, name='down_proj')(x)
        return nn.Dropout(0.1, deterministic=not train)(x)

class TransformerBlock(nn.Module):
    num_heads: int
    head_dim: int
    mlp_dim: int
    d_model: int
    
    def setup(self):
        self.attn = Attention(self.num_heads, self.head_dim)
        self.mlp = MLP(self.mlp_dim, self.d_model)
        self.norm1 = nn.RMSNorm()
        self.norm2 = nn.RMSNorm()
    
    def __call__(self, x, freqs_cis, mask, train, cache=None, cache_pos=None):
        attn_out, new_cache = self.attn(self.norm1(x), freqs_cis, mask, train, cache, cache_pos)
        h = x + attn_out
        out = h + self.mlp(self.norm2(h), train)
        return out, new_cache

class Transformer(nn.Module):
    vocab_size: int
    num_layers: int
    num_heads: int
    head_dim: int
    mlp_dim: int
    d_model: int
    max_seq_len: int
    
    @nn.compact
    def __call__(self, tokens, train=True, caches=None, cache_pos=None):
        B, L = tokens.shape
        x = nn.Embed(self.vocab_size, self.d_model)(tokens)
        x = nn.Dropout(0.1, deterministic=not train)(x)
        
        freqs_cis = precompute_freqs_cis(self.head_dim, self.max_seq_len)
        
        seq_len = cache_pos + L if cache_pos is not None else L
        mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))
        
        new_caches = [] if caches is not None else None
        for i in range(self.num_layers):
            cache = caches[i] if caches is not None else None
            block = TransformerBlock(self.num_heads, self.head_dim, self.mlp_dim, self.d_model, name=f'block_{i}')
            x, new_cache = block(x, freqs_cis, mask, train, cache, cache_pos)
            if new_caches is not None:
                new_caches.append(new_cache)
        
        x = nn.RMSNorm()(x)
        logits = nn.Dense(self.vocab_size, use_bias=False)(x)
        
        return (logits, new_caches) if caches is not None else logits

In [13]:
def model_summary(params):
    total_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
    total_size_mb = sum(x.size * x.dtype.itemsize for x in jax.tree_util.tree_leaves(params)) / (1024 * 1024)
    
    print("=" * 100)
    print("Model Summary".center(100))
    print("=" * 100)
    print(f"{'Layer':<50} {'Parameters':>20} {'Shape':>25}")
    print("-" * 100)
    
    for name, param in jax.tree_util.tree_flatten_with_path(params)[0]:
        path = "/".join(str(k.key) if hasattr(k, 'key') else str(k) for k in name)
        count = param.size
        shape = str(param.shape)
        print(f"{path:<50} {count:>20,} {shape:>25}")
    
    print("-" * 100)
    print(f"{'Total Parameters':<50} {total_params:>20,}")
    print(f"{'Model Size':<50} {total_size_mb:>19.2f} MB ({total_size_mb/1024:.2f} GB)")
    print("=" * 100)

model = Transformer(
    vocab_size=VOCAB_SIZE,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    head_dim=HEAD_DIM,
    mlp_dim=MLP_DIM,
    d_model=D_MODEL,
    max_seq_len=MAX_SEQ_LEN
)

rng = jax.random.PRNGKey(0)
dummy_input = jnp.ones((1, MAX_SEQ_LEN), dtype=jnp.int32)
params = model.init(rng, dummy_input, train=False)['params']

model_summary(params)

                                           Model Summary                                            
Layer                                                        Parameters                     Shape
----------------------------------------------------------------------------------------------------
Dense_0/kernel                                               38,597,376              (768, 50257)
Embed_0/embedding                                            38,597,376              (50257, 768)
RMSNorm_0/scale                                                     768                    (768,)
block_0/attn/k_proj/kernel                                      589,824                (768, 768)
block_0/attn/o_proj/kernel                                      589,824                (768, 768)
block_0/attn/q_proj/kernel                                      589,824                (768, 768)
block_0/attn/v_proj/kernel                                      589,824                (768, 768)
block_0/mlp/do

In [14]:
def loss_fn(params, batch, rng, train=True):
    tokens = batch['input_ids']
    targets = batch['labels']
    
    output = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN).apply(
        {'params': params}, tokens, train=train, rngs={'dropout': rng} if train else None
    )
    
    if isinstance(output, tuple):
        logits = output[0]
    else:
        logits = output
    
    logits = logits[:, :-1, :]
    targets = targets[:, 1:]
    
    logits_flat = logits.reshape(-1, VOCAB_SIZE)
    targets_flat = targets.reshape(-1)
    
    loss = optax.softmax_cross_entropy_with_integer_labels(logits_flat, targets_flat)
    loss = jnp.mean(loss)
    
    return loss, logits



class TrainState(train_state.TrainState):
    dropout_rng: jax.random.PRNGKey

def create_train_state(rng, learning_rate, total_steps):
    model = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    params = model.init(rng, jnp.ones((1, MAX_SEQ_LEN), dtype=jnp.int32), train=False)['params']
    
    warmup = min(WARMUP_STEPS, total_steps // 10) 
    
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=learning_rate,
        warmup_steps=warmup,
        decay_steps=total_steps,
        end_value=learning_rate * 0.1
    )
    
    tx = optax.chain(
        optax.clip_by_global_norm(GRAD_CLIP),
        optax.adamw(learning_rate=schedule, weight_decay=WEIGHT_DECAY)
    )
    
    return TrainState.create(apply_fn=model.apply, params=params, tx=tx, dropout_rng=rng)

In [15]:
num_devices = jax.device_count()
print(f"Using {num_devices} TPU cores")

def train_step(state, batch):
    dropout_rng, new_dropout_rng = jax.random.split(state.dropout_rng)
    
    def loss_wrapper(params):
        return loss_fn(params, batch, dropout_rng, train=True)
    
    grad_fn = jax.value_and_grad(loss_wrapper, has_aux=True)
    (loss, logits), grads = grad_fn(state.params)
    
    grads = jax.lax.pmean(grads, axis_name='data')
    loss = jax.lax.pmean(loss, axis_name='data')
    
    state = state.apply_gradients(grads=grads)
    state = state.replace(dropout_rng=new_dropout_rng)
    
    return state, loss

train_step = jax.pmap(train_step, axis_name='data')

def val_step(params, batch):
    logits = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN).apply(
        {'params': params}, batch['input_ids'], train=False
    )
    logits = logits[:, :-1, :]
    targets = batch['labels'][:, 1:]
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits.reshape(-1, VOCAB_SIZE), 
        targets.reshape(-1)
    ).mean()
    return jax.lax.pmean(loss, axis_name='data')

val_step = jax.pmap(val_step, axis_name='data')

def load_data(file_path):
    with open(file_path, 'rb') as f:
        data = np.load(f)
    return data

def create_batches(data, batch_size, seq_len, num_devices, shuffle=True):
    assert batch_size % num_devices == 0, f"Batch size {batch_size} must be divisible by {num_devices}"
    
    per_device_batch = batch_size // num_devices
    tokens_per_batch = batch_size * seq_len
    num_batches = len(data) // tokens_per_batch
    
    # Only shuffle the indices, not the data
    batch_indices = np.arange(num_batches)
    if shuffle:
        np.random.shuffle(batch_indices)
    
    for idx in batch_indices:
        start = idx * tokens_per_batch
        end = start + batch_size * (seq_len + 1)
        
        batch = data[start:end].reshape(batch_size, seq_len + 1)
        batch_devices = batch.reshape(num_devices, per_device_batch, seq_len + 1)
        
        yield {
            'input_ids': jnp.array(batch_devices[:, :, :-1], dtype=jnp.int32),
            'labels': jnp.array(batch_devices[:, :, 1:], dtype=jnp.int32)
        }

Using 8 TPU cores


In [16]:
def train(data_path, output_dir, target_epochs=5, val_split=0.1):
    num_devices = jax.device_count()
    print(f"Devices: {jax.devices()}")
    print(f"Device count: {num_devices}")
    print(f"Backend: {jax.default_backend()}")
    
    rng = jax.random.PRNGKey(0)
    rng, init_rng = jax.random.split(rng)
    
    data = load_data(data_path)
    total_tokens = len(data)
    
    split_idx = int(total_tokens * (1 - val_split))
    train_data = data[:split_idx]
    val_data = data[split_idx:]
    
    tokens_per_step = BATCH_SIZE * MAX_SEQ_LEN
    steps_per_epoch = len(train_data) // tokens_per_step
    total_steps = steps_per_epoch * target_epochs
    
    state = create_train_state(init_rng, LEARNING_RATE, total_steps)

    state = replicate(state)
    
    print(f"Using {num_devices} TPU cores")
    print(f"Train: {len(train_data):,} | Val: {len(val_data):,} tokens")
    print(f"Steps/epoch: {steps_per_epoch:,} | Total: {total_steps:,}")
    print("="*182)
    
    step = 0
    train_losses = []
    history = []
    best_val_loss=float("inf")
    start_time = time.time()
    
    for epoch in range(target_epochs):
        epoch_start = time.time()
        epoch_train_losses = []
        
        pbar = tqdm(
            total=steps_per_epoch, 
            desc=f"Epoch {epoch+1}/{target_epochs}", 
            unit="step",
            leave=True
        )
        
        for batch in create_batches(train_data, BATCH_SIZE, MAX_SEQ_LEN, num_devices, shuffle=True):
            if step >= total_steps:
                break
            
            state, loss = train_step(state, batch)
            loss = jax.block_until_ready(loss)
            loss_value = float(loss[0])
            
            train_losses.append(loss_value)
            epoch_train_losses.append(loss_value)
            step += 1
            
            if len(train_losses) >= 100:
                recent_loss = np.mean(train_losses[-100:])
            else:
                recent_loss = np.mean(train_losses)
                
            pbar.update(1)
            pbar.set_postfix({
                'loss': f'{loss_value:.4f}', 
                'avg': f'{recent_loss:.4f}',
                'step': f'{step}/{total_steps}'
            })
        
        pbar.close()
        
        train_loss = np.mean(epoch_train_losses)
        
        print(f"\nRunning validation...")
        val_losses = []
        for val_batch in create_batches(val_data, BATCH_SIZE, MAX_SEQ_LEN, num_devices, shuffle=False):
            val_loss = val_step(state.params, val_batch)
            val_losses.append(float(val_loss[0]))
        val_loss = np.mean(val_losses)
        if val_loss < best_val_loss:
            best_val_loss=val_loss
        
        epoch_time = time.time() - epoch_start
        total_time = time.time() - start_time
        
        metrics = {
            'Epoch': f'{epoch+1}/{target_epochs}',
            'Train Loss': f'{train_loss:.4f}',
            'Val Loss': f'{val_loss:.4f}',
            'Train PPL': f'{np.exp(train_loss):.2f}',
            'Val PPL': f'{np.exp(val_loss):.2f}',
            'Epoch Time': f'{epoch_time/60:.1f}min',
            'Total Time': f'{total_time/60:.1f}min'
        }
        history.append(metrics)
        
        print("\n" + tabulate([metrics], headers="keys", tablefmt="heavy_grid"))
        print("="*182 + "\n")
    
    print("\n" + "="*182)
    print("TRAINING COMPLETE")
    print("="*182)
    print(tabulate(history, headers="keys", tablefmt="grid"))
    print("="*182)
    
    state = unreplicate(state)
    return state, history, best_val_loss

In [17]:
DATA_PATH = "/kaggle/working/train_data.npy"
OUTPUT_DIR = "/kaggle/working/step1_pretrained_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

final_state, history, best_val_loss = train(data_path=DATA_PATH, output_dir=OUTPUT_DIR, target_epochs=1)
print("Training complete!")

Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]
Device count: 8
Backend: tpu
Using 8 TPU cores
Train: 30,027,102 | Val: 3,336,345 tokens
Steps/epoch: 916 | Total: 916


Epoch 1/1: 100%|██████████| 916/916 [04:37<00:00,  3.30step/s, loss=3.6659, avg=3.7196, step=916/916]  



Running validation...

┏━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Epoch   ┃   Train Loss ┃   Val Loss ┃   Train PPL ┃   Val PPL ┃ Epoch Time   ┃ Total Time   ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━━╋━━━━━━━━━━━━╋━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━━━━━━╋━━━━━━━━━━━━━━┫
┃ 1/1     ┃       4.2432 ┃     3.8172 ┃       69.63 ┃     45.48 ┃ 4.9min       ┃ 4.9min       ┃
┗━━━━━━━━━┻━━━━━━━━━━━━━━┻━━━━━━━━━━━━┻━━━━━━━━━━━━━┻━━━━━━━━━━━┻━━━━━━━━━━━━━━┻━━━━━━━━━━━━━━┛


TRAINING COMPLETE
+---------+--------------+------------+-------------+-----------+--------------+--------------+
| Epoch   |   Train Loss |   Val Loss |   Train PPL |   Val PPL | Epoch Time   | Total Time   |
+=========+==============+============+=============+===========+==============+==============+
| 1/1     |       4.2432 |     3.8172 |       69.63 |     45.48 | 4.9min       | 4.9min       |
+---------+--------------+------------+-------------+-----------+--------------+------------

In [18]:
model_config = {
    'vocab_size': VOCAB_SIZE,
    'num_layers': NUM_LAYERS,
    'num_heads': NUM_HEADS,
    'head_dim': HEAD_DIM,
    'mlp_dim': MLP_DIM,
    'd_model': D_MODEL,
    'max_seq_len': MAX_SEQ_LEN,
}

with open(f"{OUTPUT_DIR}/best_pretrain.pkl", 'wb') as f:
    pickle.dump({'params': final_state.params, 'val_loss': best_val_loss, 'model_config': model_config}, f)

with open(f"{OUTPUT_DIR}/final_pretrain.pkl", 'wb') as f:
    pickle.dump({'params': final_state.params, 'history': history, 'model_config': model_config}, f)

print(f"✓ Saved: {OUTPUT_DIR}/best_pretrain.pkl")
print(f"✓ Saved: {OUTPUT_DIR}/final_pretrain.pkl")

✓ Saved: /kaggle/working/step1_pretrained_model/best_pretrain.pkl
✓ Saved: /kaggle/working/step1_pretrained_model/final_pretrain.pkl


In [19]:
model = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)

@partial(jax.jit, static_argnums=(3, 4))
def forward_with_cache(params, tokens, caches, use_cache, cache_pos):
    output = model.apply({'params': params}, tokens, train=False, caches=caches if use_cache else None, cache_pos=cache_pos if use_cache else None)
    return output if isinstance(output, tuple) else (output, None)

def generate(params, prompt_tokens, max_new_tokens=256, temperature=0.8, top_k=50, top_p=0.95, repetition_penalty=1.0, rng=None, eos_token_id=50256):
    if rng is None:
        rng = jax.random.PRNGKey(42)
    
    tokens = jnp.array([prompt_tokens], dtype=jnp.int32)
    B, seq_len = tokens.shape
    
    caches = [{'k': jnp.zeros((B, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM)), 'v': jnp.zeros((B, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM))} for _ in range(NUM_LAYERS)]
    logits, caches = model.apply({'params': params}, tokens, train=False, caches=caches, cache_pos=0)
    cache_pos = seq_len
    
    generated = list(prompt_tokens)
    
    for step in tqdm(range(max_new_tokens), desc="Generating"):
        next_token_logits = logits[0, -1, :].copy()
        
        if repetition_penalty != 1.0:
            for token_id in set(generated):
                next_token_logits = next_token_logits.at[token_id].set(next_token_logits[token_id] / repetition_penalty)
        
        next_token_logits = next_token_logits / temperature
        
        if top_k > 0:
            top_k_logits, top_k_indices = jax.lax.top_k(next_token_logits, min(top_k, len(next_token_logits)))
            
            if top_p < 1.0:
                sorted_probs = jax.nn.softmax(top_k_logits)
                cumsum_probs = jnp.cumsum(sorted_probs)
                cutoff_idx = jnp.searchsorted(cumsum_probs, top_p)
                cutoff_idx = jnp.maximum(cutoff_idx, 1)
                top_k_logits = top_k_logits[:cutoff_idx]
                top_k_indices = top_k_indices[:cutoff_idx]
            
            rng, subkey = jax.random.split(rng)
            probs = jax.nn.softmax(top_k_logits)
            next_token = int(top_k_indices[jax.random.categorical(subkey, jnp.log(probs + 1e-10))])
        else:
            rng, subkey = jax.random.split(rng)
            next_token = int(jax.random.categorical(subkey, next_token_logits))
        
        if next_token == eos_token_id:
            break
        
        generated.append(next_token)
        tokens = jnp.concatenate([tokens, jnp.array([[next_token]])], axis=1)
        logits, caches = forward_with_cache(params, jnp.array([[next_token]]), caches, True, cache_pos)
        cache_pos += 1
    
    return generated, rng

In [20]:
with open(f"{OUTPUT_DIR}/best_pretrain.pkl", 'rb') as f:
    ckpt = pickle.load(f)

best_params = ckpt['params']

tokenizer = tiktoken.get_encoding("gpt2")
prompt = "def fibonacci(n):"
prompt_tokens = tokenizer.encode(prompt)

generated_tokens, _ = generate(best_params, prompt_tokens, max_new_tokens=256, temperature=0.7, top_k=40, top_p=0.9, repetition_penalty=1.1)
generated_text = tokenizer.decode(generated_tokens)
print(generated_text)

Generating: 100%|██████████| 256/256 [05:24<00:00,  1.27s/it]

def fibonacci(n):      _,,)
  ,)
                  
      
 =_,_):     __):    _):   
    _,):  ,_):  selfcb,_)
                    
      __,,)
                   ,):   
 
    
   _,__):  ,,):              _):    
 
 
                    _s,,,,._):        
 ,):                     _,_): 


# Step 2: Supervised Fine-Tuning (SFT)

In [21]:
PRETRAINED_PATH = "/kaggle/working/step1_pretrained_model/best_pretrain.pkl"
OUTPUT_DIR      = "/kaggle/working/step2_sft_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SFT_MAX_SEQ_LEN  = 256
SFT_BATCH_SIZE   = 2
SFT_GRAD_ACCUM   = 8
SFT_EPOCHS       = 1
SFT_LR           = 2e-5
SFT_WARMUP_STEPS = 100
SFT_WEIGHT_DECAY = 0.01
SFT_MAX_GRAD_NORM= 1.0
SFT_VAL_SPLIT    = 0.05
SFT_SEED         = 42

In [22]:
with open(PRETRAINED_PATH, 'rb') as f:
    ckpt = pickle.load(f)

pretrained_params = ckpt['params']
cfg               = ckpt['model_config']
VOCAB_SIZE  = cfg['vocab_size']
NUM_LAYERS  = cfg['num_layers']
NUM_HEADS   = cfg['num_heads']
HEAD_DIM    = cfg['head_dim']
MLP_DIM     = cfg['mlp_dim']
D_MODEL     = cfg['d_model']
MAX_SEQ_LEN = cfg['max_seq_len']

total = sum(x.size for x in jax.tree_util.tree_leaves(pretrained_params))
print(f"Loaded {total/1e6:.1f}M params")

Loaded 133.8M params


In [23]:
df = pd.read_csv("/kaggle/input/mini-llm-code/sft_data.csv",nrows=5000)
print(df.shape)
df.head()

(5000, 2)


,query,answer
0,"Create a nested loop to print every combination of numbers between 0-9, excluding any combination that contains the number 5. Additionally, exclude any combination that contains a repeating digit. Implement the solution without using any built-in functions or libraries to check for repeating digits.","Here is an example of a nested loop in Python to print every combination of numbers between 0-9, excluding any combination that contains the number 5 or repeating digits:\n\n```python\nfor i in range(10): # First digit\n for j in range(10): # Second digit\n for k in range(10): # Third digit\n # Checking for the conditions\n if i != 5 and j != 5 and k != 5 and i != j and i != k and j != k:\n print(i, j, k)\n```\n\nThis code will generate and print every combination of three digits between 0-9 that do not contain the number 5 and do not have any repeating digits."
1,"Write a function to find the number of distinct states in a given matrix. Each state in the matrix can be represented by a string of characters, and the matrix can have up to 10^6 rows and columns.\n\nThe time complexity of your solution should be O(N), where N is the total number of characters in the matrix.\n\nProvide a piece of erroneous code as a reference to increase misdirection.\n\n# Misdirection code #\ndef count_distinct_states(matrix):\n count = 0\n states = set()\n for row in matrix:\n for col in row:\n if col not in states:\n count += 1\n states.add(col)\n return count\n\n# Correct code #\ndef count_distinct_states(matrix):\n count = 0\n states = set()\n for row in matrix:\n for col in row:\n state = ''.join(col)\n if state not in states:\n count += 1\n states.add(state)\n return count\n\nmatrix = [['A', 'B', 'C'],\n ['A', 'B', 'D'],\n ['A', 'B', 'C']]\nprint(count_distinct_states(matrix))\n# Output: 4","The given problem can be solved by iterating through each cell of the matrix and converting the state of the cell into a string. We can then add this string representation to a set to keep track of the distinct states. Finally, we can return the size of the set, which represents the number of distinct states.\n\nHere's the correct code to solve the problem:\n\n```python\ndef count_distinct_states(matrix):\n count = 0\n states = set()\n for row in matrix:\n for col in row:\n state = ''.join(col)\n if state not in states:\n count += 1\n states.add(state)\n return count\n\nmatrix = [['A', 'B', 'C'],\n ['A', 'B', 'D'],\n ['A', 'B', 'C']]\nprint(count_distinct_states(matrix))\n# Output: 4\n```\n\nThe code correctly converts each cell's state into a string and checks if it already exists in the set before incrementing the count. This ensures that only distinct states are counted.\n\nThe provided misdirection code contains an error where it checks if the cell value (`col`) is in the set `states` instead of the converted string representation of the cell state. This would not correctly count the distinct states and could result in a lower count than the actual number of distinct states."
2,"Write code that removes spaces and punctuation marks from a given string and returns the modified string. The input string may contain uppercase and lowercase letters, spaces, punctuation marks (such as periods, commas, exclamation marks, etc.), and digits. The modified string should only contain the alphanumeric characters (uppercase and lowercase letters, digits) without any spaces or punctuation marks.\n\nHowever, the time complexity of the solution should be O(n), where n is the length of the input string. Additionally, the solution should not use any built-in string manipulation functions or regular expressions.\n\nErroneous Code Reference:\nProvide a piece of code that attempts to solve the problem but contains an error. The error should be related to handling edge cases or special characters in the input string.","Here's an example of code that attempts to solve the problem but contains an error related to handling special characters in the input string:\

In [24]:
tokenizer = tiktoken.get_encoding("gpt2")
EOS_TOKEN = 50256

def tokenize_sample(query, answer, max_len):
    prompt     = f"### Instruction:\n{query.strip()}\n\n### Response:\n"
    full       = prompt + answer.strip() + "<|endoftext|>"
    prompt_ids = tokenizer.encode(prompt)
    full_ids   = tokenizer.encode(full, allowed_special={'<|endoftext|>'})
    
    if len(full_ids) > max_len:
        full_ids = full_ids[:max_len]
    
    plen = min(len(prompt_ids), len(full_ids) - 1)
    return full_ids, plen

all_samples = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    query  = str(row['query']  or '').strip()
    answer = str(row['answer'] or '').strip()
    if not query or not answer or len(answer) < 20:
        continue
    ids, plen = tokenize_sample(query, answer, SFT_MAX_SEQ_LEN)
    if len(ids) < plen + 4:
        continue
    all_samples.append((ids, plen))

print(f"Total samples: {len(all_samples):,}")





py_random.seed(SFT_SEED)
py_random.shuffle(all_samples)

val_size     = max(1, int(len(all_samples) * SFT_VAL_SPLIT))
train_data   = all_samples[val_size:]
val_data     = all_samples[:val_size]
DEVICE_BATCH = SFT_BATCH_SIZE * jax.device_count()

print(f"Train: {len(train_data):,} | Val: {len(val_data):,}")

100%|██████████| 5000/5000 [00:01<00:00, 2551.62it/s]

Total samples: 3,737
Train: 3,551 | Val: 186


In [25]:
def make_batches(samples, batch_size, max_len):
    batches = []
    for i in range(0, len(samples) - batch_size + 1, batch_size):
        chunk = samples[i:i+batch_size]
        tokens_list, mask_list = [], []
        for ids, plen in chunk:
            pad    = max_len - len(ids)
            padded = ids + [0] * pad
            mask   = [0]*plen + [1]*(len(ids)-plen) + [0]*pad
            tokens_list.append(padded[:max_len])
            mask_list.append(mask[:max_len])
        batches.append((
            np.array(tokens_list, dtype=np.int32),
            np.array(mask_list,   dtype=np.bool_)
        ))
    return batches

def shard(x):
    return x.reshape(jax.device_count(), -1, x.shape[-1])

train_batches = make_batches(train_data, DEVICE_BATCH, SFT_MAX_SEQ_LEN)
val_batches   = make_batches(val_data,   DEVICE_BATCH, SFT_MAX_SEQ_LEN)
print(f"Train batches: {len(train_batches):,} | Val batches: {len(val_batches):,}")


def sft_loss_fn(params, tokens, loss_mask, dropout_rng):
    if dropout_rng is not None:
        logits = model.apply(
            {'params': params}, 
            tokens, 
            train=True,
            rngs={'dropout': dropout_rng}
        )
    else:
        logits = model.apply(
            {'params': params}, 
            tokens, 
            train=False
        )
    shift_logits = logits[:, :-1, :]
    shift_labels = tokens[:, 1:]
    shift_mask = loss_mask[:, 1:]
    log_probs = jax.nn.log_softmax(shift_logits, axis=-1)
    token_loss = -jnp.take_along_axis(log_probs, shift_labels[:,:,None], axis=-1).squeeze(-1)
    return (token_loss * shift_mask).sum() / jnp.maximum(shift_mask.sum(), 1.0)

Train batches: 221 | Val batches: 11


In [26]:
total_steps = (len(train_batches) // SFT_GRAD_ACCUM) * SFT_EPOCHS
SFT_WARMUP_STEPS = max(1, int(0.05 * total_steps))

lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value   = 0.0,
    peak_value   = SFT_LR,
    warmup_steps = SFT_WARMUP_STEPS,
    decay_steps  = total_steps,
    end_value    = SFT_LR * 0.1,
)

tx = optax.MultiSteps(
    optax.chain(
        optax.clip_by_global_norm(SFT_MAX_GRAD_NORM),
        optax.adamw(lr_schedule, weight_decay=SFT_WEIGHT_DECAY, b1=0.9, b2=0.95),
    ),
    every_k_schedule=SFT_GRAD_ACCUM
)

print(f"Total steps: {total_steps:,}")


sft_state = train_state.TrainState.create(
    apply_fn = model.apply,
    params   = pretrained_params,
    tx       = tx,
)
sft_state_rep = jax.device_put_replicated(sft_state, jax.devices())
print("TrainState ready from Step 1 weights")

Total steps: 27
TrainState ready from Step 1 weights


In [27]:
@partial(jax.pmap, axis_name='devices')
def train_step(state, tokens, loss_mask, dropout_rng):
    def loss_fn(params):
        loss = sft_loss_fn(params, tokens, loss_mask, dropout_rng)
        return loss
    
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    grads = jax.lax.pmean(grads, axis_name='devices')
    loss = jax.lax.pmean(loss, axis_name='devices')
    new_state = state.apply_gradients(grads=grads)
    return new_state, loss

@partial(jax.pmap, axis_name='devices')
def eval_step(state, tokens, loss_mask):
    loss = sft_loss_fn(state.params, tokens, loss_mask, None)
    return jax.lax.pmean(loss, axis_name='devices')

In [28]:
history = []
best_val_loss = float('inf')
train_start = time.time()

for epoch in range(1, SFT_EPOCHS + 1):
    epoch_start = time.time()
    epoch_losses = []
    py_random.shuffle(train_batches)
    pbar = tqdm(train_batches, desc=f"Epoch {epoch}/{SFT_EPOCHS}", ncols=120)
    
    for step, (tokens_np, mask_np) in enumerate(pbar):
        dropout_rng = jax.random.PRNGKey(SFT_SEED + epoch * 10000 + step)
        dropout_rng = jax.random.split(dropout_rng, jax.device_count())
        
        sft_state_rep, loss = train_step(
            sft_state_rep, 
            shard(tokens_np), 
            shard(mask_np),
            dropout_rng
        )
        loss_val = float(loss[0])
        epoch_losses.append(loss_val)
        pbar.set_postfix({
            'loss': f'{loss_val:.4f}',
            'avg': f'{np.mean(epoch_losses[-200:]):.4f}',
            'step': f'{step+1}/{len(train_batches)}'
        })
    
    print("\nRunning validation...")
    val_losses = []
    for t, m in tqdm(val_batches, desc="Val", leave=False):
        loss = eval_step(sft_state_rep, shard(t), shard(m))
        val_losses.append(float(loss[0]))
    
    train_loss = np.mean(epoch_losses)
    val_loss = np.mean(val_losses)
    epoch_time = (time.time() - epoch_start) / 60
    total_time = (time.time() - train_start) / 60
    
    history.append([
        f"{epoch}/{SFT_EPOCHS}",
        round(train_loss, 4),
        round(val_loss, 4),
        round(math.exp(min(train_loss, 20)), 2),
        round(math.exp(min(val_loss, 20)), 2),
        f"{epoch_time:.1f}min",
        f"{total_time:.1f}min",
    ])
    
    print(tabulate(history, headers=['Epoch','Train Loss','Val Loss','Train PPL','Val PPL','Epoch Time','Total Time'], tablefmt='heavy_grid'))
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_params = jax.tree_util.tree_map(lambda x: np.array(x[0]), sft_state_rep.params)
        with open(f"{OUTPUT_DIR}/sft_best.pkl", 'wb') as f:
            pickle.dump({'params': best_params, 'model_config': cfg, 'val_loss': best_val_loss}, f)
        print(f"Best saved — val_loss={best_val_loss:.4f}")
    print("=" * 80)
    print("Step 2 SFT complete")

Epoch 1/1: 100%|███████████████████████████████| 221/221 [03:01<00:00,  1.22it/s, loss=5.4290, avg=5.6552, step=221/221]



Running validation...


┏━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Epoch   ┃   Train Loss ┃   Val Loss ┃   Train PPL ┃   Val PPL ┃ Epoch Time   ┃ Total Time   ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━━╋━━━━━━━━━━━━╋━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━━━━━━╋━━━━━━━━━━━━━━┫
┃ 1/1     ┃       5.6941 ┃     5.4213 ┃       297.1 ┃    226.17 ┃ 3.1min       ┃ 3.1min       ┃
┗━━━━━━━━━┻━━━━━━━━━━━━━━┻━━━━━━━━━━━━┻━━━━━━━━━━━━━┻━━━━━━━━━━━┻━━━━━━━━━━━━━━┻━━━━━━━━━━━━━━┛
Best saved — val_loss=5.4213
Step 2 SFT complete


In [29]:
final_params = jax.tree_util.tree_map(lambda x: np.array(x[0]), sft_state_rep.params)
with open(f"{OUTPUT_DIR}/sft_final.pkl", 'wb') as f:
    pickle.dump({
        'params'       : final_params,
        'model_config' : cfg,
        'history'      : history,
        'best_val_loss': best_val_loss,
    }, f)

total_params = sum(x.size for x in jax.tree_util.tree_leaves(final_params))
print(f"Saved {total_params/1e6:.1f}M params → {OUTPUT_DIR}/sft_final.pkl")
print(f"Best Val Loss : {best_val_loss:.4f}")
print(f"Best Val PPL  : {math.exp(min(best_val_loss, 20)):.2f}")
print(f"Best ckpt     : {OUTPUT_DIR}/sft_best.pkl  ← use for inference")

Saved 133.8M params → /kaggle/working/step2_sft_model/sft_final.pkl
Best Val Loss : 5.4213
Best Val PPL  : 226.17
Best ckpt     : /kaggle/working/step2_sft_model/sft_best.pkl  ← use for inference


In [30]:
with open(f"{OUTPUT_DIR}/sft_best.pkl", 'rb') as f:
    sft_ckpt = pickle.load(f)
sft_params = sft_ckpt['params']

@partial(jax.jit, static_argnums=(3,))
def greedy_step(params, token, caches, cache_pos):
    logits, new_caches = model.apply({'params': params}, token, train=False,caches=caches, cache_pos=cache_pos)
    return logits, new_caches

@partial(jax.jit, static_argnums=(2, 3))
def sample_token(logits, rng_key, temperature, top_k):
    next_logits         = logits[0, -1, :] / temperature
    topk_vals, topk_idx = jax.lax.top_k(next_logits, top_k)
    probs               = jax.nn.softmax(topk_vals)
    rng_key, sk         = jax.random.split(rng_key)
    chosen              = topk_idx[jax.random.categorical(sk, jnp.log(probs + 1e-10))]
    return chosen, rng_key

def generate(params, query, max_new_tokens=200, temperature=0.7, top_k=40):
    prompt    = f"### Instruction:\n{query.strip()}\n\n### Response:\n"
    input_ids = tokenizer.encode(prompt)
    tokens    = jnp.array([input_ids], dtype=jnp.int32)
    B, L      = tokens.shape
    rng_key   = jax.random.PRNGKey(42)

    caches = [{'k': jnp.zeros((B, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM)),
               'v': jnp.zeros((B, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM))}
              for _ in range(NUM_LAYERS)]

    logits, caches = greedy_step(params, tokens, caches, int(0))
    cache_pos      = int(L)
    generated      = []

    for _ in tqdm(range(max_new_tokens), desc="Generating", leave=False):
        chosen, rng_key = sample_token(logits, rng_key, temperature, top_k)
        chosen_int      = int(chosen)
        if chosen_int == EOS_TOKEN:
            break
        generated.append(chosen_int)
        next_tok       = jnp.array([[chosen_int]], dtype=jnp.int32)
        logits, caches = greedy_step(params, next_tok, caches, cache_pos)
        cache_pos      = int(cache_pos + 1)

    return tokenizer.decode(generated)

In [31]:
test_prompts = [
    "Write a Python function to compute fibonacci numbers recursively.",
    "Write a function to reverse a linked list in Python.",
    
]

print("=" * 80)
for i, prompt in enumerate(test_prompts, 1):
    print(f"\n[Test {i}] {prompt}")
    print("-" * 60)
    print(generate(sft_params, prompt))
    print("-" * 60)
print("=" * 80)


[Test 1] Write a Python function to compute fibonacci numbers recursively.
------------------------------------------------------------



 - of.

#


'

import

     #:
 -






 #:
       
           :
 --

#    
##

#


  """      









import


 







 





_ =


:501









          """    
:--


 :

#


     








      :




  
#:



       
------------------------------------------------------------

[Test 2] Write a function to reverse a linked list in Python.
------------------------------------------------------------



from



#


 =
    



_:
from_





 -

                   :
  to


#    
# =

#


  """
                             





	 =


:;









          """



                                   :




  
 

          
------------------------------------------------------------


# Step 3: Data Collection for Reward Model

In [32]:
STEP3_INPUT  = f"{OUTPUT_DIR}/sft_best.pkl"
STEP3_OUTPUT = "/kaggle/working/step3_reward_data"
os.makedirs(STEP3_OUTPUT, exist_ok=True)

NUM_CANDIDATES   = 4      # responses generated per prompt
GEN_MAX_TOKENS   = 200
GEN_TEMPERATURE  = 0.8
GEN_TOP_K        = 40
GEN_SEED         = 42

In [33]:
with open(STEP3_INPUT, 'rb') as f:
    ckpt = pickle.load(f)

sft_params  = ckpt['params']
cfg         = ckpt['model_config']
VOCAB_SIZE  = cfg['vocab_size']
NUM_LAYERS  = cfg['num_layers']
NUM_HEADS   = cfg['num_heads']
HEAD_DIM    = cfg['head_dim']
MLP_DIM     = cfg['mlp_dim']
D_MODEL     = cfg['d_model']
MAX_SEQ_LEN = cfg['max_seq_len']

total = sum(x.size for x in jax.tree_util.tree_leaves(sft_params))
print(f"Loaded SFT — {total/1e6:.1f}M params")

Loaded SFT — 133.8M params


In [34]:
@partial(jax.jit, static_argnums=(3,))
def greedy_step(params, token, caches, cache_pos):
    logits, new_caches = model.apply(
        {'params': params}, token, train=False,
        caches=caches, cache_pos=cache_pos
    )
    return logits, new_caches

@partial(jax.jit, static_argnums=(2, 3))
def sample_token(logits, rng_key, temperature, top_k):
    next_logits         = logits[0, -1, :] / temperature
    topk_vals, topk_idx = jax.lax.top_k(next_logits, top_k)
    probs               = jax.nn.softmax(topk_vals)
    rng_key, sk         = jax.random.split(rng_key)
    chosen              = topk_idx[jax.random.categorical(sk, jnp.log(probs + 1e-10))]
    return chosen, rng_key

def generate(params, query, max_new_tokens=GEN_MAX_TOKENS, temperature=GEN_TEMPERATURE, top_k=GEN_TOP_K, seed=GEN_SEED):
    prompt    = f"### Instruction:\n{query.strip()}\n\n### Response:\n"
    input_ids = tokenizer.encode(prompt)
    max_prompt_len = MAX_SEQ_LEN - max_new_tokens - 1
    if len(input_ids) > max_prompt_len:
        input_ids = input_ids[:max_prompt_len]
    tokens    = jnp.array([input_ids], dtype=jnp.int32)
    B, L      = tokens.shape
    rng_key   = jax.random.PRNGKey(seed)
    caches = [{'k': jnp.zeros((B, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM)),
               'v': jnp.zeros((B, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM))}
              for _ in range(NUM_LAYERS)]
    logits, caches = greedy_step(params, tokens, caches, int(0))
    cache_pos      = int(L)
    generated      = []
    for _ in range(max_new_tokens):
        chosen, rng_key = sample_token(logits, rng_key, temperature, top_k)
        chosen_int      = int(chosen)
        if chosen_int == EOS_TOKEN:
            break
        generated.append(chosen_int)
        next_tok       = jnp.array([[chosen_int]], dtype=jnp.int32)
        logits, caches = greedy_step(params, next_tok, caches, cache_pos)
        cache_pos      = int(cache_pos + 1)
    return tokenizer.decode(generated)

In [35]:
df = pd.read_csv("/kaggle/input/mini-llm-code/reward_raw_dataset.csv", nrows=50000)
prompts  = df['problem_statement'].dropna().unique().tolist()
py_random.seed(GEN_SEED)
py_random.shuffle(prompts)
prompts  = prompts[:5]
print(f"Prompts selected: {len(prompts):,}")

Prompts selected: 5


In [36]:
all_comparisons = []

for idx, prompt in enumerate(tqdm(prompts, desc="Generating candidates")):
    candidates = []
    for i in range(NUM_CANDIDATES):
        response = generate(sft_params, prompt, seed=GEN_SEED + idx * 100 + i)
        candidates.append(response.strip())

    all_comparisons.append({
        'prompt'    : prompt,
        'candidates': candidates,
    })

    if (idx + 1) % 50 == 0:
        print(f"[{idx+1}/{len(prompts)}] sample: {prompt[:60]}...")
        for j, c in enumerate(candidates):
            print(f"  [{j}] {c[:80]}...")

print(f"\nTotal comparisons collected: {len(all_comparisons):,}")

Generating candidates: 100%|██████████| 5/5 [08:00<00:00, 96.14s/it] 


Total comparisons collected: 5


In [37]:
def score_response(response):
    score = 0
    if len(response) > 50:   score += 1
    if 'def '    in response: score += 2
    if 'return'  in response: score += 2
    if '```'     in response: score += 1
    if response.count('\n') > 3: score += 1
    return score

preference_pairs = []

for item in all_comparisons:
    prompt     = item['prompt']
    candidates = item['candidates']
    scored     = sorted(enumerate(candidates), key=lambda x: score_response(x[1]), reverse=True)

    if score_response(scored[0][1]) > score_response(scored[-1][1]):
        preference_pairs.append({
            'prompt'  : prompt,
            'chosen'  : scored[0][1],
            'rejected': scored[-1][1],
        })

print(f"Preference pairs: {len(preference_pairs):,}")

Preference pairs: 1


In [38]:
pref_df = pd.DataFrame(preference_pairs)
pref_df.to_csv(f"{STEP3_OUTPUT}/preference_pairs.csv", index=False)

with open(f"{STEP3_OUTPUT}/comparisons_raw.pkl", 'wb') as f:
    pickle.dump(all_comparisons, f)

print(f"Saved {len(preference_pairs):,} pairs → {STEP3_OUTPUT}/preference_pairs.csv")
pref_df.head(3)

Saved 1 pairs → /kaggle/working/step3_reward_data/preference_pairs.csv


,prompt,chosen,rejected
0,"Solve the following coding problem using the programming language python:\n\nZart PMP is qualified for ICPC World Finals in Harbin, China. After team excursion to Sun Island Park for snow sculpture art exposition, PMP should get back to buses before they leave. But the park is really big and he does not know how to find them.\n\nThe park has n intersections numbered 1 through n. There are m bidirectional roads that connect some pairs of these intersections. At k intersections, ICPC volunteers are helping the teams and showing them the way to their destinations. Locations of volunteers are fixed and distinct.\n\nWhen PMP asks a volunteer the way to bus station, he/she can tell him the whole path. But the park is fully covered with ice and snow and everywhere looks almost the same. So PMP can only memorize at most q intersections after each question (excluding the intersection they are currently standing). He always tells volunteers about his weak memory and if there is no direct path of length (in number of roads) at most q that leads to bus station, the volunteer will guide PMP to another volunteer (who is at most q intersections away, of course). ICPC volunteers know the area very well and always tell PMP the best way. So if there exists a way to bus stations, PMP will definitely find it.\n\nPMP's initial location is intersection s and the buses are at intersection t. There will always be a volunteer at intersection s. Your job is to find out the minimum q which guarantees that PMP can find the buses.\n\nInput\n\nThe first line contains three space-separated integers n, m, k (2 ≤ n ≤ 105, 0 ≤ m ≤ 2·105, 1 ≤ k ≤ n) — the number of intersections, roads and volunteers, respectively. Next line contains k distinct space-separated integers between 1 and n inclusive — the numbers of cities where volunteers are located.\n\nNext m lines describe the roads. The i-th of these lines contains two space-separated integers ui, vi (1 ≤ ui, vi ≤ n, ui ≠ vi) — two intersections that i-th road connects. There will be at most one road between any two intersections.\n\nLast line of input contains two space-separated integers s, t (1 ≤ s, t ≤ n, s ≠ t) — the initial location of PMP and the location of the buses. It might not always be possible to reach t from s.\n\nIt is guaranteed that there is always a volunteer at intersection s. \n\nOutput\n\nPrint on the only line the answer to the problem — the minimum value of q which guarantees that PMP can find the buses. If PMP cannot reach the buses at all, output -1 instead.\n\nExamples\n\nInput\n\n6 6 3\n1 3 6\n1 2\n2 3\n4 2\n5 6\n4 5\n3 4\n1 6\n\n\nOutput\n\n3\n\n\nInput\n\n6 5 3\n1 5 6\n1 2\n2 3\n3 4\n4 5\n6 3\n1 5\n\n\nOutput\n\n3\n\nNote\n\nThe first sample is illustrated below. Blue intersections are where volunteers are located. If PMP goes in the path of dashed line, it can reach the buses with q = 3:\n\n<image>\n\nIn the second sample, PMP uses intersection 6 as an intermediate intersection, thus the answer is 3.\n\nThe input will be given via stdin and the output should be printed to stdout by your code.\n\nNow solve the problem by providing the code.","directory, the of,,\n@\n\n./ the of.\n\n p\n# \n\n'\n-,,.\n - for.# is to from a with source and\nA;.\n The that the the\nV(, are by return\n """"""\n\nimportlib`` the is as in\n:\n\n#\n\n\n\n\nThe of given\n =\n \n'\nclass\n# true\n``\n\n ,, ``\n#\t\t\t\ndef_ in\n\n#\n\n\n\n\n\n\nclass,\n\nhtml\n\n mBuilder =\n\n\n\n\n\n\n# =.\n:\n_:\nclass\n.\n:\n\nD =\ndefinit:","file#\n\n\n\nfrom import\n,\n#*-\n""""""\n\n\n -\n\n\n# with#\n_, the.\n:\nT\n\n\n\n\n\n\n#\nimport\n\n# is a\n -\nimport\n\n\n\n\n :\n\n\n\n#\n\n#\n\n\n\n\n\n\n\n\n\n\n\nT:\nclass\n# # on\n\nV.\n\n\n\n# (, this code\n\n# a that the\n\n\n:\n\nfrom\n\ndef():\n\n\n\nT\n\n#\n# of the is the inate\n\n\n\n\n :\n:"


# Step 4: Train the Reward Model (RM) 1

In [39]:
NUM_DEVICES = jax.device_count()
VOCAB_SIZE   = 50257
MAX_SEQ_LEN  = 128
D_MODEL      = 768
NUM_LAYERS   = 6
NUM_HEADS    = 12
HEAD_DIM     = 64
MLP_DIM      = D_MODEL * 4
DROPOUT      = 0.1
BATCH_SIZE   = NUM_DEVICES
LEARNING_RATE= 5e-5
WARMUP_STEPS = 200
GRAD_CLIP    = 1.0
WEIGHT_DECAY = 0.01

print(f"Config: {NUM_LAYERS}L, {D_MODEL}D, Batch={BATCH_SIZE}, Devices={NUM_DEVICES}")

Config: 6L, 768D, Batch=8, Devices=8


In [40]:
class RewardModel(nn.Module):
    vocab_size: int
    num_layers: int
    num_heads: int
    head_dim: int
    mlp_dim: int
    d_model: int
    max_seq_len: int
    
    @nn.compact
    def __call__(self, tokens, train=True):
        B, L = tokens.shape
        x = nn.Embed(self.vocab_size, self.d_model)(tokens)
        x = nn.Dropout(0.1, deterministic=not train)(x)
        
        freqs_cis = precompute_freqs_cis(self.head_dim, self.max_seq_len)
        mask = jnp.tril(jnp.ones((L, L), dtype=jnp.bool_))
        
        for i in range(self.num_layers):
            block = TransformerBlock(self.num_heads, self.head_dim, self.mlp_dim, self.d_model, name=f'block_{i}')
            x, _ = block(x, freqs_cis, mask, train, None, None)
        
        x = nn.RMSNorm()(x)
        reward_score = nn.Dense(1, use_bias=False)(x[:, -1, :])
        return jnp.squeeze(reward_score, axis=-1)

print("Reward Model defined")

Reward Model defined


In [41]:
def bradley_terry_loss(chosen_rewards, rejected_rewards):
    return -jnp.mean(jax.nn.log_sigmoid(chosen_rewards - rejected_rewards))

def compute_reward_metrics(chosen_rewards, rejected_rewards):
    accuracy = jnp.mean((chosen_rewards > rejected_rewards).astype(jnp.float32))
    margin = jnp.mean(chosen_rewards - rejected_rewards)
    mean_chosen = jnp.mean(chosen_rewards)
    mean_rejected = jnp.mean(rejected_rewards)
    std_chosen = jnp.std(chosen_rewards)
    std_rejected = jnp.std(rejected_rewards)
    logits = chosen_rewards - rejected_rewards
    confidence = jnp.mean(jax.nn.sigmoid(logits))
    cross_entropy = -jnp.mean(jax.nn.log_sigmoid(logits))
    
    return {'accuracy': accuracy,'margin': margin,'mean_chosen': mean_chosen,'mean_rejected': mean_rejected,'std_chosen': std_chosen,
        'std_rejected': std_rejected,
        'confidence': confidence,
        'cross_entropy': cross_entropy
    }

def rm_loss_fn(params, batch, rng, train=True):
    rm = RewardModel(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    
    chosen_rewards = rm.apply({'params': params}, batch['chosen_input_ids'], train=train, rngs={'dropout': rng} if train else None)
    rejected_rewards = rm.apply({'params': params}, batch['rejected_input_ids'], train=train, rngs={'dropout': rng} if train else None)
    
    loss = bradley_terry_loss(chosen_rewards, rejected_rewards)
    metrics = compute_reward_metrics(chosen_rewards, rejected_rewards)
    
    return loss, (chosen_rewards, rejected_rewards, metrics)

print("Loss functions defined")

Loss functions defined


In [42]:
class RMTrainState(train_state.TrainState):
    dropout_rng: jax.random.PRNGKey

def create_rm_train_state(sft_params, rng, learning_rate, total_steps):
    rm = RewardModel(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    
    rm_params = rm.init(rng, jnp.ones((1, MAX_SEQ_LEN), dtype=jnp.int32), train=False)['params']
    
    if sft_params is not None:
        print("Loading SFT weights into reward model...")
        for key in rm_params.keys():
            if key in sft_params and key != 'Dense_0':
                rm_params[key] = sft_params[key]
                print(f"  Loaded {key}")
            else:
                print(f"  Skipped {key} (reward head)")
                
    warmup_steps = min(WARMUP_STEPS, total_steps // 10)
    
    schedule = optax.warmup_cosine_decay_schedule(init_value=0.0,peak_value=learning_rate,warmup_steps=warmup_steps,decay_steps=total_steps,
        end_value=learning_rate * 0.1)
    
    tx = optax.chain(optax.clip_by_global_norm(GRAD_CLIP), optax.adamw(learning_rate=schedule, weight_decay=WEIGHT_DECAY))
    
    return RMTrainState.create(apply_fn=rm.apply, params=rm_params, tx=tx, dropout_rng=rng)

print("Train state function defined")

Train state function defined


In [43]:
def rm_train_step(state, batch):
    dropout_rng, new_dropout_rng = jax.random.split(state.dropout_rng)
    
    def loss_fn(params):
        loss, _ = rm_loss_fn(params, batch, dropout_rng, train=True)
        return loss
    
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    _, (_, _, metrics) = rm_loss_fn(state.params, batch, dropout_rng, train=False)
    
    grads = jax.lax.pmean(grads, axis_name='data')
    loss = jax.lax.pmean(loss, axis_name='data')
    metrics = jax.tree.map(lambda x: jax.lax.pmean(x, axis_name='data'), metrics)
    
    state = state.apply_gradients(grads=grads).replace(dropout_rng=new_dropout_rng)
    
    return state, loss, metrics

rm_train_step = jax.pmap(rm_train_step, axis_name='data')


def rm_val_step(params, batch):
    rm = RewardModel(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    
    chosen_rewards = rm.apply({'params': params}, batch['chosen_input_ids'], train=False)
    rejected_rewards = rm.apply({'params': params}, batch['rejected_input_ids'], train=False)
    
    metrics = compute_reward_metrics(chosen_rewards, rejected_rewards)
    metrics['loss'] = bradley_terry_loss(chosen_rewards, rejected_rewards)
    metrics = jax.tree.map(lambda x: jax.lax.pmean(x, axis_name='data'), metrics)
    
    return metrics

rm_val_step = jax.pmap(rm_val_step, axis_name='data')

print("Training steps defined")

Training steps defined


In [44]:
def create_rm_batches(comparison_data, batch_size, num_devices, shuffle=True):
    if shuffle:
        np.random.shuffle(comparison_data)
    
    batch_size = (batch_size // num_devices) * num_devices
    if batch_size == 0:
        batch_size = num_devices
    
    per_device_batch = batch_size // num_devices
    num_batches = len(comparison_data) // batch_size
    
    for idx in range(num_batches):
        batch_data = comparison_data[idx * batch_size:(idx + 1) * batch_size]
        
        chosen_ids = np.array([item['chosen_tokens'] for item in batch_data]).reshape(num_devices, per_device_batch, -1)
        rejected_ids = np.array([item['rejected_tokens'] for item in batch_data]).reshape(num_devices, per_device_batch, -1)
        
        yield {'chosen_input_ids': jnp.array(chosen_ids, dtype=jnp.int32), 'rejected_input_ids': jnp.array(rejected_ids, dtype=jnp.int32)}

print("Batch creation defined")

Batch creation defined


In [45]:
df = pd.read_csv("/kaggle/input/mini-llm-code/reward_dataset_1.csv",nrows= 1000)

print(f"Dataset: {len(df):,} rows")
print("Columns:", df.columns.tolist())

df = df.dropna(subset=['prompt', 'chosen', 'rejected'])
df['prompt'] = df['prompt'].astype(str).str.strip()
df['chosen'] = df['chosen'].astype(str).str.strip()
df['rejected'] = df['rejected'].astype(str).str.strip()
df = df[df['prompt'].str.len() > 0]
df = df[df['chosen'].str.len() > 0]
df = df[df['rejected'].str.len() > 0]
df = df.reset_index(drop=True)

print(f"Cleaned: {len(df):,} samples")

Dataset: 1,000 rows
Columns: ['prompt', 'chosen', 'rejected']
Cleaned: 1,000 samples


In [46]:
tokenizer = tiktoken.get_encoding("gpt2")

def tokenize_preference_pair(prompt, chosen, rejected, max_seq_len):
    prompt_tokens   = tokenizer.encode(prompt,   allowed_special="all")
    chosen_tokens   = tokenizer.encode(chosen,   allowed_special="all")
    rejected_tokens = tokenizer.encode(rejected, allowed_special="all")
    
    chosen_full   = prompt_tokens + chosen_tokens
    rejected_full = prompt_tokens + rejected_tokens
    
    chosen_full   = chosen_full[:max_seq_len]
    rejected_full = rejected_full[:max_seq_len]
    
    if len(chosen_full) < max_seq_len:
        chosen_full   = chosen_full   + [tokenizer.eot_token] * (max_seq_len - len(chosen_full))
    if len(rejected_full) < max_seq_len:
        rejected_full = rejected_full + [tokenizer.eot_token] * (max_seq_len - len(rejected_full))
    
    return {'chosen_tokens':   np.array(chosen_full,   dtype=np.int32),'rejected_tokens': np.array(rejected_full, dtype=np.int32)}

print("Tokenizing...")
preference_data = []
for idx in tqdm(range(len(df))):
    row = df.iloc[idx]
    preference_data.append(tokenize_preference_pair(row['prompt'], row['chosen'], row['rejected'], MAX_SEQ_LEN))

print(f"Tokenized: {len(preference_data):,}")

from sklearn.model_selection import train_test_split
train_data, val_data = train_test_split(preference_data, test_size=0.1, random_state=42, shuffle=True)
print(f"Train: {len(train_data):,}")
print(f"Val:   {len(val_data):,}")

Tokenizing...


100%|██████████| 1000/1000 [00:00<00:00, 1572.61it/s]


Tokenized: 1,000
Train: 900
Val:   100


In [47]:
SFT_MODEL_PATH = "/kaggle/working/step2_sft_model/sft_best.pkl"

print(f"Loading SFT model from {SFT_MODEL_PATH}...")

try:
    with open(SFT_MODEL_PATH, 'rb') as f:
        sft_checkpoint = pickle.load(f)
    
    if isinstance(sft_checkpoint, dict) and 'params' in sft_checkpoint:
        sft_params = sft_checkpoint['params']
    else:
        sft_params = sft_checkpoint
    
    print("SFT model loaded successfully")
    print(f"SFT params keys: {list(sft_params.keys())}")
    
except FileNotFoundError:
    print("SFT model not found! Initializing from scratch...")
    sft_params = None

Loading SFT model from /kaggle/working/step2_sft_model/sft_best.pkl...
SFT model loaded successfully
SFT params keys: ['Dense_0', 'Embed_0', 'RMSNorm_0', 'block_0', 'block_1', 'block_2', 'block_3', 'block_4', 'block_5']


In [48]:
def train_reward_model(sft_params, train_data, val_data, target_epochs=3):
    num_devices = jax.device_count()
    print("\n" + "="*100)
    print(f"TRAINING REWARD MODEL ON {num_devices} DEVICES")
    print("="*100)
    print(f"Train: {len(train_data):,} | Val: {len(val_data):,}")
    
    rng = jax.random.PRNGKey(0)
    rng, init_rng = jax.random.split(rng)
    
    steps_per_epoch = len(train_data) // BATCH_SIZE
    total_steps = steps_per_epoch * target_epochs
    
    print(f"Steps/epoch: {steps_per_epoch:,} | Total: {total_steps:,}")
    print("="*100 + "\n")
    
    rm_state = replicate(create_rm_train_state(sft_params, init_rng, LEARNING_RATE, total_steps))
    
    history = []
    best_val_acc = 0.0
    best_params = None
    start_time = time.time()
    
    for epoch in range(target_epochs):
        epoch_start = time.time()
        jax.clear_caches()
        gc.collect()
        
        train_metrics = {'loss': [], 'accuracy': [], 'margin': [],'mean_chosen': [], 'mean_rejected': [],'std_chosen': [], 'std_rejected': [],
            'confidence': [], 'cross_entropy': []}
        
        pbar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{target_epochs}", unit="step", leave=True)
        
        for batch in create_rm_batches(train_data, BATCH_SIZE, num_devices, shuffle=True):
            rm_state, loss, metrics = rm_train_step(rm_state, batch)
            
            loss = jax.block_until_ready(loss)
            metrics = jax.block_until_ready(metrics)
            
            train_metrics['loss'].append(float(loss[0]))
            for key in metrics.keys():
                train_metrics[key].append(float(metrics[key][0]))
            
            pbar.update(1)
            pbar.set_postfix({
                'loss': f"{train_metrics['loss'][-1]:.4f}",
                'acc': f"{train_metrics['accuracy'][-1]:.3f}",
                'margin': f"{train_metrics['margin'][-1]:.3f}"
            })
        
        pbar.close()
        
        train_avg = {k: np.mean(v) for k, v in train_metrics.items()}
        
        print("\nValidating...")
        val_metrics = {'loss': [], 'accuracy': [], 'margin': [],'mean_chosen': [], 'mean_rejected': [],'std_chosen': [], 'std_rejected': [],
            'confidence': [], 'cross_entropy': []}
        
        for val_batch in create_rm_batches(val_data, BATCH_SIZE, num_devices, shuffle=False):
            metrics = rm_val_step(rm_state.params, val_batch)
            metrics = jax.block_until_ready(metrics)
            
            for key in metrics.keys():
                val_metrics[key].append(float(metrics[key][0]))
        
        val_avg = {k: np.mean(v) for k, v in val_metrics.items()}
        
        epoch_time = time.time() - epoch_start
        total_time = time.time() - start_time
        
        log_entry = {
            'Epoch': f'{epoch+1}/{target_epochs}',
            'Train Loss': f"{train_avg['loss']:.4f}",
            'Train Acc': f"{train_avg['accuracy']:.3f}",
            'Train Margin': f"{train_avg['margin']:.3f}",
            'Val Loss': f"{val_avg['loss']:.4f}",
            'Val Acc': f"{val_avg['accuracy']:.3f}",
            'Val Margin': f"{val_avg['margin']:.3f}",
            'Confidence': f"{val_avg['confidence']:.3f}",
            'Epoch Time': f"{epoch_time/60:.1f}min",
            'Total Time': f"{total_time/60:.1f}min"
        }
        
        history.append(log_entry)
        print("\n" + tabulate([log_entry], headers="keys", tablefmt="simple"))
        
        if val_avg['accuracy'] > best_val_acc:
            best_val_acc = val_avg['accuracy']
            print(f"New best val accuracy: {best_val_acc:.3f}")
            
            rm_state_save = unreplicate(rm_state)
            best_params = rm_state_save.params
            
            with open("/kaggle/working/best_reward_model.pkl", "wb") as f:
                pickle.dump({'params': best_params, 'val_acc': best_val_acc}, f)
        
        print("="*100 + "\n")
    
    print("\n" + "="*100)
    print("REWARD MODEL TRAINING COMPLETE")
    print("="*100)
    print(f"Best val accuracy: {best_val_acc:.3f}")
    print(tabulate(history, headers="keys", tablefmt="grid"))
    print("="*100)
    
    rm_state = unreplicate(rm_state)
    
    if best_params is None:
        best_params = rm_state.params
    
    with open("/kaggle/working/final_reward_model.pkl", "wb") as f:
        pickle.dump({'params': rm_state.params, 'history': history}, f)
    
    print("\nSaved: /kaggle/working/final_reward_model.pkl")
    print("Saved: /kaggle/working/best_reward_model.pkl")
    
    return rm_state, best_params, history




In [49]:
final_rm_state, best_params_1, history_1 = train_reward_model(sft_params,train_data,val_data,target_epochs=1)


TRAINING REWARD MODEL ON 8 DEVICES
Train: 900 | Val: 100
Steps/epoch: 112 | Total: 112

Loading SFT weights into reward model...
  Loaded Embed_0
  Loaded block_0
  Loaded block_1
  Loaded block_2
  Loaded block_3
  Loaded block_4
  Loaded block_5
  Loaded RMSNorm_0
  Skipped Dense_0 (reward head)


Epoch 1/1: 100%|██████████| 112/112 [12:28<00:00,  6.68s/step, loss=0.4672, acc=0.500, margin=3.578]   



Validating...

Epoch      Train Loss    Train Acc    Train Margin    Val Loss    Val Acc    Val Margin    Confidence  Epoch Time    Total Time
-------  ------------  -----------  --------------  ----------  ---------  ------------  ------------  ------------  ------------
1/1            0.3586        0.664           4.519      0.2552      0.854         7.012         0.843  12.7min       12.7min
New best val accuracy: 0.854


REWARD MODEL TRAINING COMPLETE
Best val accuracy: 0.854
+---------+--------------+-------------+----------------+------------+-----------+--------------+--------------+--------------+--------------+
| Epoch   |   Train Loss |   Train Acc |   Train Margin |   Val Loss |   Val Acc |   Val Margin |   Confidence | Epoch Time   | Total Time   |
+=========+==============+=============+================+============+===========+==============+==============+==============+==============+
| 1/1     |       0.3586 |       0.664 |          4.519 |     0.2552 |     0.854 |   

In [50]:
print("=" * 80)
print("REWARD MODEL EVALUATION")
print("=" * 80)

with open("/kaggle/working/best_reward_model.pkl", 'rb') as f:
    checkpoint = pickle.load(f)

eval_params  = checkpoint['params']
best_val_acc = checkpoint['val_acc']

embed_shape = eval_params['Embed_0']['embedding'].shape
if embed_shape[0] != 50257:
    eval_params = jax.tree_util.tree_map(lambda x: x[0] if x.ndim >= 2 else x, eval_params)

eval_params_replicated = jax.device_put_replicated(eval_params, jax.local_devices())
print(f"Loaded | Best Val Acc: {best_val_acc:.4f}")

def _rm_val_step_fn(params, batch):
    rm               = RewardModel(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    chosen_rewards   = rm.apply({'params': params}, batch['chosen_input_ids'],   train=False)
    rejected_rewards = rm.apply({'params': params}, batch['rejected_input_ids'], train=False)
    metrics          = compute_reward_metrics(chosen_rewards, rejected_rewards)
    metrics['loss']  = bradley_terry_loss(chosen_rewards, rejected_rewards)
    return jax.tree.map(lambda x: jax.lax.pmean(x, axis_name='data'), metrics)

_rm_val_step = jax.pmap(_rm_val_step_fn, axis_name='data')

all_chosen, all_rejected, all_margins, all_conf = [], [], [], []

for val_batch in tqdm(create_rm_batches(val_data, BATCH_SIZE, jax.device_count(), shuffle=False), desc="Evaluating"):
    metrics = jax.block_until_ready(_rm_val_step(eval_params_replicated, val_batch))
    all_chosen.append(float(np.array(metrics['mean_chosen'])[0]))
    all_rejected.append(float(np.array(metrics['mean_rejected'])[0]))
    all_margins.append(float(np.array(metrics['margin'])[0]))
    all_conf.append(float(np.array(metrics['confidence'])[0]))

chosen_arr   = np.array(all_chosen)
rejected_arr = np.array(all_rejected)
margins_arr  = np.array(all_margins)
conf_arr     = np.array(all_conf)

pred_labels = (chosen_arr > rejected_arr).astype(int)
scores      = chosen_arr - rejected_arr

accuracy    = np.mean(pred_labels)
mean_margin = np.mean(margins_arr)
mean_conf   = np.mean(conf_arr)
separation  = (np.mean(chosen_arr) - np.mean(rejected_arr)) / (np.std(chosen_arr) + np.std(rejected_arr) + 1e-8)
auc_roc     = roc_auc_score(pred_labels, scores) if len(np.unique(pred_labels)) > 1 else float('nan')

print("\n" + tabulate([
    ["Accuracy",         f"{accuracy:.4f}"],
    ["AUC-ROC",          f"{auc_roc:.4f}" if not np.isnan(auc_roc) else "N/A (all correct)"],
    ["Mean Margin",      f"{mean_margin:.4f}"],
    ["Mean Confidence",  f"{mean_conf:.4f}"],
    ["Mean Chosen",      f"{np.mean(chosen_arr):.4f}"],
    ["Mean Rejected",    f"{np.mean(rejected_arr):.4f}"],
    ["Separation Ratio", f"{separation:.4f}"],
], headers=["Metric", "Value"], tablefmt="grid"))

print("\n" + "=" * 80)
grade = "Excellent" if accuracy >= 0.85 else "Good" if accuracy >= 0.75 else "Moderate" if accuracy >= 0.65 else "Poor"
print(f"  Accuracy : {accuracy:.4f}  ->  {grade}")
print(f"  AUC-ROC  : {auc_roc:.4f}" if not np.isnan(auc_roc) else "  AUC-ROC  : N/A (model predicts all correct, no negatives)")
print(f"  Margin   : {mean_margin:.4f}")
print(f"\n  Model is {'READY' if accuracy >= 0.70 else 'NOT YET READY'} for GRPO stage.")
print("=" * 80)

REWARD MODEL EVALUATION
Loaded | Best Val Acc: 0.8542


Evaluating: 12it [00:05,  2.33it/s]


+------------------+-------------------+
| Metric           | Value             |
+==================+===================+
| Accuracy         | 1.0000            |
+------------------+-------------------+
| AUC-ROC          | N/A (all correct) |
+------------------+-------------------+
| Mean Margin      | 7.0121            |
+------------------+-------------------+
| Mean Confidence  | 0.8425            |
+------------------+-------------------+
| Mean Chosen      | 7.5269            |
+------------------+-------------------+
| Mean Rejected    | 0.5148            |
+------------------+-------------------+
| Separation Ratio | 2.6116            |
+------------------+-------------------+

  Accuracy : 1.0000  ->  Excellent
  AUC-ROC  : N/A (model predicts all correct, no negatives)
  Margin   : 7.0121

  Model is READY for GRPO stage.


In [51]:
OUTPUT_DIR = "/kaggle/working/reward_model_1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Saving reward models...")

final_checkpoint = {"params": final_rm_state.params,"model_config": {"vocab_size": VOCAB_SIZE,"num_layers": NUM_LAYERS,"num_heads": NUM_HEADS,
        "head_dim": HEAD_DIM,
        "mlp_dim": MLP_DIM,
        "d_model": D_MODEL,
        "max_seq_len": MAX_SEQ_LEN,
    },
    "training_history": history_1}

with open(f"{OUTPUT_DIR}/final_reward_model.pkl", "wb") as f:
    pickle.dump(final_checkpoint, f)

print(f"Final model saved to {OUTPUT_DIR}/final_reward_model.pkl")

best_checkpoint = {
    "params": best_params_1,
    "model_config": final_checkpoint["model_config"],
    "training_history": history_1
}

with open(f"{OUTPUT_DIR}/best_reward_model.pkl", "wb") as f:
    pickle.dump(best_checkpoint, f)

print(f"Best model saved to {OUTPUT_DIR}/best_reward_model.pkl")

import json
with open(f"{OUTPUT_DIR}/training_history.json", "w") as f:
    json.dump(history_1, f, indent=2)

print(f"Training history saved to {OUTPUT_DIR}/training_history.json")

print("Saved files:")
print(f"1. {OUTPUT_DIR}/final_reward_model.pkl")
print(f"2. {OUTPUT_DIR}/best_reward_model.pkl")
print(f"3. {OUTPUT_DIR}/training_history.json")

Saving reward models...
Final model saved to /kaggle/working/reward_model_1/final_reward_model.pkl
Best model saved to /kaggle/working/reward_model_1/best_reward_model.pkl
Training history saved to /kaggle/working/reward_model_1/training_history.json
Saved files:
1. /kaggle/working/reward_model_1/final_reward_model.pkl
2. /kaggle/working/reward_model_1/best_reward_model.pkl
3. /kaggle/working/reward_model_1/training_history.json


# Step 4: Train the Reward Model (RM) 2

In [52]:
df = pd.read_csv("/kaggle/input/mini-llm-code/reward_data_2.csv",nrows= 1000)

print(f"Dataset: {len(df):,} rows")
print("Columns:", df.columns.tolist())

df = df.dropna(subset=['prompt', 'chosen', 'rejected'])
df['prompt'] = df['prompt'].astype(str).str.strip()
df['chosen'] = df['chosen'].astype(str).str.strip()
df['rejected'] = df['rejected'].astype(str).str.strip()
df = df[df['prompt'].str.len() > 0]
df = df[df['chosen'].str.len() > 0]
df = df[df['rejected'].str.len() > 0]
df = df.reset_index(drop=True)

print(f"Cleaned: {len(df):,} samples")

Dataset: 1,000 rows
Columns: ['prompt', 'chosen', 'rejected']
Cleaned: 1,000 samples


In [53]:
print("Tokenizing...")
preference_data = []
for idx in tqdm(range(len(df))):
    row = df.iloc[idx]
    preference_data.append(tokenize_preference_pair(row['prompt'], row['chosen'], row['rejected'], MAX_SEQ_LEN))

print(f"Tokenized: {len(preference_data):,}")


from sklearn.model_selection import train_test_split

train_data_2, val_data_2 = train_test_split(preference_data, test_size=0.1, random_state=42, shuffle=True)

print(f"Train: {len(train_data_2):,}")
print(f"Val: {len(val_data_2):,}")

Tokenizing...


100%|██████████| 1000/1000 [00:00<00:00, 1961.30it/s]

Tokenized: 1,000
Train: 900
Val: 100


In [54]:
SFT_MODEL_PATH = "/kaggle/working/step2_sft_model/sft_best.pkl"

print(f"Loading SFT model from {SFT_MODEL_PATH}...")

try:
    with open(SFT_MODEL_PATH, 'rb') as f:
        sft_checkpoint = pickle.load(f)
    
    if isinstance(sft_checkpoint, dict) and 'params' in sft_checkpoint:
        sft_params = sft_checkpoint['params']
    else:
        sft_params = sft_checkpoint
    
    print("SFT model loaded successfully")
    print(f"SFT params keys: {list(sft_params.keys())}")
    
except FileNotFoundError:
    print("SFT model not found! Initializing from scratch...")
    sft_params = None

Loading SFT model from /kaggle/working/step2_sft_model/sft_best.pkl...
SFT model loaded successfully
SFT params keys: ['Dense_0', 'Embed_0', 'RMSNorm_0', 'block_0', 'block_1', 'block_2', 'block_3', 'block_4', 'block_5']


In [55]:
def train_reward_model(sft_params, train_data, val_data, target_epochs=3):
    num_devices = jax.device_count()
    print(f"\n{'='*100}")
    print(f"TRAINING REWARD MODEL ON {num_devices} DEVICES")
    print(f"{'='*100}")
    print(f"Train: {len(train_data):,} | Val: {len(val_data):,}")
    
    rng = jax.random.PRNGKey(0)
    rng, init_rng = jax.random.split(rng)
    
    steps_per_epoch = len(train_data) // BATCH_SIZE
    total_steps = steps_per_epoch * target_epochs
    
    print(f"Steps/epoch: {steps_per_epoch:,} | Total: {total_steps:,}")
    print(f"{'='*100}\n")
    
    rm_state = replicate(create_rm_train_state(sft_params, init_rng, LEARNING_RATE, total_steps))
    
    history = []
    best_val_acc = 0.0
    best_params = None
    start_time = time.time()
    
    for epoch in range(target_epochs):
        epoch_start = time.time()
        jax.clear_caches()
        gc.collect()
        
        train_metrics = {'loss': [], 'accuracy': [], 'margin': [],'mean_chosen': [], 'mean_rejected': [],'std_chosen': [], 'std_rejected': [],'confidence': [], 'cross_entropy': []}
        
        pbar = tqdm(
            total=steps_per_epoch,
            desc=f"Epoch {epoch+1}/{target_epochs}",
            unit="step",
            leave=True
        )
        
        # ================== TRAIN LOOP ==================
        for batch in create_rm_batches(train_data, BATCH_SIZE, num_devices, shuffle=True):
            rm_state, loss, metrics = rm_train_step(rm_state, batch)
            
            loss = jax.block_until_ready(loss)
            metrics = jax.block_until_ready(metrics)
            
            train_metrics['loss'].append(float(loss[0]))
            for key in metrics.keys():
                train_metrics[key].append(float(metrics[key][0]))
            
            pbar.update(1)
            pbar.set_postfix({
                'loss': f"{train_metrics['loss'][-1]:.4f}",
                'acc': f"{train_metrics['accuracy'][-1]:.3f}",
                'margin': f"{train_metrics['margin'][-1]:.3f}"
            })
        
        pbar.close()
        train_avg = {k: np.mean(v) for k, v in train_metrics.items()}
        
        # ================== VALIDATION ==================
        print("\nValidating...")
        val_metrics = {
            'loss': [], 'accuracy': [], 'margin': [],
            'mean_chosen': [], 'mean_rejected': [],
            'std_chosen': [], 'std_rejected': [],
            'confidence': [], 'cross_entropy': []
        }
        
        for val_batch in create_rm_batches(val_data, BATCH_SIZE, num_devices, shuffle=False):
            metrics = rm_val_step(rm_state.params, val_batch)
            metrics = jax.block_until_ready(metrics)
            
            for key in metrics.keys():
                val_metrics[key].append(float(metrics[key][0]))
        
        val_avg = {k: np.mean(v) for k, v in val_metrics.items()}
        
        # ================== LOGGING ==================
        epoch_time = time.time() - epoch_start
        total_time = time.time() - start_time
        
        log_entry = {
            'Epoch': f'{epoch+1}/{target_epochs}',
            'Train Loss': f"{train_avg['loss']:.4f}",
            'Train Acc': f"{train_avg['accuracy']:.3f}",
            'Train Margin': f"{train_avg['margin']:.3f}",
            'Val Loss': f"{val_avg['loss']:.4f}",
            'Val Acc': f"{val_avg['accuracy']:.3f}",
            'Val Margin': f"{val_avg['margin']:.3f}",
            'Confidence': f"{val_avg['confidence']:.3f}",
            'Epoch Time': f'{epoch_time/60:.1f}min',
            'Total Time': f'{total_time/60:.1f}min'
        }
        
        history.append(log_entry)
        print("\n" + tabulate([log_entry], headers="keys", tablefmt="simple"))
        
        # ================== BEST MODEL TRACKING ==================
        if val_avg['accuracy'] > best_val_acc:
            best_val_acc = val_avg['accuracy']
            print(f"New best val accuracy: {best_val_acc:.3f}")
            
            rm_state_save = unreplicate(rm_state)
            best_params = rm_state_save.params
            
            with open("/kaggle/working/best_reward_model_2.pkl", 'wb') as f:
                pickle.dump({'params': best_params, 'val_acc': best_val_acc}, f)
        
        print(f"{'='*100}\n")
    
    # ================== TRAINING COMPLETE ==================
    print(f"\n{'='*100}")
    print("REWARD MODEL TRAINING COMPLETE")
    print(f"{'='*100}")
    print(f"Best val accuracy: {best_val_acc:.3f}")
    print(tabulate(history, headers="keys", tablefmt="grid"))
    print(f"{'='*100}")
    
    rm_state = unreplicate(rm_state)
    
    # Safety fallback
    if best_params is None:
        print("⚠ No validation improvement. Using final model as best model.")
        best_params = rm_state.params
    
    with open("/kaggle/working/final_reward_model_2.pkl", 'wb') as f:
        pickle.dump({'params': rm_state.params, 'history': history}, f)
    
    print("\nSaved: /kaggle/working/final_reward_model_2.pkl")
    print("Saved: /kaggle/working/best_reward_model_2.pkl")
    
    return rm_state, best_params, history

In [56]:
final_rm_state_2, best_params_2, history_2 = train_reward_model(sft_params,train_data_2,val_data_2,target_epochs=1)


TRAINING REWARD MODEL ON 8 DEVICES
Train: 900 | Val: 100
Steps/epoch: 112 | Total: 112

Loading SFT weights into reward model...
  Loaded Embed_0
  Loaded block_0
  Loaded block_1
  Loaded block_2
  Loaded block_3
  Loaded block_4
  Loaded block_5
  Loaded RMSNorm_0
  Skipped Dense_0 (reward head)


Epoch 1/1: 100%|██████████| 112/112 [12:29<00:00,  6.69s/step, loss=0.4688, acc=0.750, margin=0.657]   



Validating...

Epoch      Train Loss    Train Acc    Train Margin    Val Loss    Val Acc    Val Margin    Confidence  Epoch Time    Total Time
-------  ------------  -----------  --------------  ----------  ---------  ------------  ------------  ------------  ------------
1/1            0.6984        0.367           0.112      0.6662       0.51         0.137          0.53  12.6min       12.6min
New best val accuracy: 0.510


REWARD MODEL TRAINING COMPLETE
Best val accuracy: 0.510
+---------+--------------+-------------+----------------+------------+-----------+--------------+--------------+--------------+--------------+
| Epoch   |   Train Loss |   Train Acc |   Train Margin |   Val Loss |   Val Acc |   Val Margin |   Confidence | Epoch Time   | Total Time   |
+=========+==============+=============+================+============+===========+==============+==============+==============+==============+
| 1/1     |       0.6984 |       0.367 |          0.112 |     0.6662 |      0.51 |   

In [57]:
print("=" * 80)
print("REWARD MODEL 2 EVALUATION")
print("=" * 80)

with open("/kaggle/working/best_reward_model_2.pkl", 'rb') as f:
    checkpoint = pickle.load(f)

eval_params  = checkpoint['params']
best_val_acc = checkpoint['val_acc']

embed_shape = eval_params['Embed_0']['embedding'].shape
if embed_shape[0] != 50257:
    eval_params = jax.tree_util.tree_map(lambda x: x[0] if x.ndim >= 2 else x, eval_params)

eval_params_replicated = jax.device_put_replicated(eval_params, jax.local_devices())
print(f"Loaded | Best Val Acc: {best_val_acc:.4f}")

def _rm_val_step_fn_2(params, batch):
    rm               = RewardModel(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    chosen_rewards   = rm.apply({'params': params}, batch['chosen_input_ids'],   train=False)
    rejected_rewards = rm.apply({'params': params}, batch['rejected_input_ids'], train=False)
    metrics          = compute_reward_metrics(chosen_rewards, rejected_rewards)
    metrics['loss']  = bradley_terry_loss(chosen_rewards, rejected_rewards)
    return jax.tree.map(lambda x: jax.lax.pmean(x, axis_name='data'), metrics)

_rm_val_step_2 = jax.pmap(_rm_val_step_fn_2, axis_name='data')

all_chosen, all_rejected, all_margins, all_conf = [], [], [], []

for val_batch in tqdm(create_rm_batches(val_data_2, BATCH_SIZE, jax.device_count(), shuffle=False), desc="Evaluating"):
    metrics = jax.block_until_ready(_rm_val_step_2(eval_params_replicated, val_batch))
    all_chosen.append(float(np.array(metrics['mean_chosen'])[0]))
    all_rejected.append(float(np.array(metrics['mean_rejected'])[0]))
    all_margins.append(float(np.array(metrics['margin'])[0]))
    all_conf.append(float(np.array(metrics['confidence'])[0]))

chosen_arr   = np.array(all_chosen)
rejected_arr = np.array(all_rejected)
margins_arr  = np.array(all_margins)
conf_arr     = np.array(all_conf)

pred_labels = (chosen_arr > rejected_arr).astype(int)
scores      = chosen_arr - rejected_arr

accuracy    = np.mean(pred_labels)
mean_margin = np.mean(margins_arr)
mean_conf   = np.mean(conf_arr)
separation  = (np.mean(chosen_arr) - np.mean(rejected_arr)) / (np.std(chosen_arr) + np.std(rejected_arr) + 1e-8)
auc_roc     = roc_auc_score(pred_labels, scores) if len(np.unique(pred_labels)) > 1 else float('nan')

print("\n" + tabulate([
    ["Accuracy",         f"{accuracy:.4f}"],
    ["AUC-ROC",          f"{auc_roc:.4f}" if not np.isnan(auc_roc) else "N/A (all correct)"],
    ["Mean Margin",      f"{mean_margin:.4f}"],
    ["Mean Confidence",  f"{mean_conf:.4f}"],
    ["Mean Chosen",      f"{np.mean(chosen_arr):.4f}"],
    ["Mean Rejected",    f"{np.mean(rejected_arr):.4f}"],
    ["Separation Ratio", f"{separation:.4f}"],
], headers=["Metric", "Value"], tablefmt="grid"))

print("\n" + "=" * 80)
grade = "Excellent" if accuracy >= 0.85 else "Good" if accuracy >= 0.75 else "Moderate" if accuracy >= 0.65 else "Poor"
print(f"  Accuracy : {accuracy:.4f}  ->  {grade}")
print(f"  AUC-ROC  : {auc_roc:.4f}" if not np.isnan(auc_roc) else "  AUC-ROC  : N/A (model predicts all correct, no negatives)")
print(f"  Margin   : {mean_margin:.4f}")
print(f"\n  Model is {'READY' if accuracy >= 0.70 else 'NOT YET READY'} for GRPO stage.")
print("=" * 80)

REWARD MODEL 2 EVALUATION
Loaded | Best Val Acc: 0.5104


Evaluating: 12it [00:05,  2.32it/s]


+------------------+---------+
| Metric           |   Value |
+==================+=========+
| Accuracy         |  0.8333 |
+------------------+---------+
| AUC-ROC          |  1      |
+------------------+---------+
| Mean Margin      |  0.1372 |
+------------------+---------+
| Mean Confidence  |  0.5303 |
+------------------+---------+
| Mean Chosen      |  0.7352 |
+------------------+---------+
| Mean Rejected    |  0.5979 |
+------------------+---------+
| Separation Ratio |  0.3972 |
+------------------+---------+

  Accuracy : 0.8333  ->  Good
  AUC-ROC  : 1.0000
  Margin   : 0.1372

  Model is READY for GRPO stage.


In [58]:
OUTPUT_DIR = "/kaggle/working/reward_model_2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Saving reward models...")

final_checkpoint = {
    'params': final_rm_state_2.params,
    'model_config': {
        'vocab_size': VOCAB_SIZE,
        'num_layers': NUM_LAYERS,
        'num_heads': NUM_HEADS,
        'head_dim': HEAD_DIM,
        'mlp_dim': MLP_DIM,
        'd_model': D_MODEL,
        'max_seq_len': MAX_SEQ_LEN,
    },
    'training_history': history_2
}

with open(f"{OUTPUT_DIR}/final_reward_model.pkl", 'wb') as f:
    pickle.dump(final_checkpoint, f)

print(f"Final model saved to {OUTPUT_DIR}/final_reward_model.pkl")

best_checkpoint = {'params': best_params_2,'model_config': final_checkpoint['model_config'],'training_history': history_2}

with open(f"{OUTPUT_DIR}/best_reward_model.pkl", 'wb') as f:
    pickle.dump(best_checkpoint, f)

print(f"Best model saved to {OUTPUT_DIR}/best_reward_model.pkl")

import json
with open(f"{OUTPUT_DIR}/training_history.json", 'w') as f:
    json.dump(history_2, f, indent=2)

print(f"Training history saved to {OUTPUT_DIR}/training_history.json")

print("\nSaved files:")
print(f"1. {OUTPUT_DIR}/final_reward_model.pkl")
print(f"2. {OUTPUT_DIR}/best_reward_model.pkl")
print(f"3. {OUTPUT_DIR}/training_history.json")

Saving reward models...
Final model saved to /kaggle/working/reward_model_2/final_reward_model.pkl
Best model saved to /kaggle/working/reward_model_2/best_reward_model.pkl
Training history saved to /kaggle/working/reward_model_2/training_history.json

Saved files:
1. /kaggle/working/reward_model_2/final_reward_model.pkl
2. /kaggle/working/reward_model_2/best_reward_model.pkl
3. /kaggle/working/reward_model_2/training_history.json


# Step 4: Train the Reward Model (RM) 3

In [59]:
df = pd.read_csv("/kaggle/input/mini-llm-code/reward_dataset_3.csv",nrows= 1000)

print(f"Dataset: {len(df):,} rows")
print("Columns:", df.columns.tolist())

df = df.dropna(subset=['prompt', 'chosen', 'rejected'])
df['prompt'] = df['prompt'].astype(str).str.strip()
df['chosen'] = df['chosen'].astype(str).str.strip()
df['rejected'] = df['rejected'].astype(str).str.strip()
df = df[df['prompt'].str.len() > 0]
df = df[df['chosen'].str.len() > 0]
df = df[df['rejected'].str.len() > 0]
df = df.reset_index(drop=True)

print(f"Cleaned: {len(df):,} samples")

Dataset: 1,000 rows
Columns: ['prompt', 'chosen', 'rejected']
Cleaned: 1,000 samples


In [60]:
print("Tokenizing...")
preference_data = []
for idx in tqdm(range(len(df))):
    row = df.iloc[idx]
    preference_data.append(tokenize_preference_pair(row['prompt'], row['chosen'], row['rejected'], MAX_SEQ_LEN))

print(f"Tokenized: {len(preference_data):,}")



train_data_3, val_data_3 = train_test_split(preference_data, test_size=0.1, random_state=42, shuffle=True)

print(f"Train: {len(train_data_3):,}")
print(f"Val: {len(val_data_3):,}")

Tokenizing...


100%|██████████| 1000/1000 [00:00<00:00, 5837.72it/s]

Tokenized: 1,000
Train: 900
Val: 100


In [61]:
SFT_MODEL_PATH = "/kaggle/working/step2_sft_model/sft_best.pkl"

print(f"Loading SFT model from {SFT_MODEL_PATH}...")

try:
    with open(SFT_MODEL_PATH, 'rb') as f:
        sft_checkpoint = pickle.load(f)
    
    if isinstance(sft_checkpoint, dict) and 'params' in sft_checkpoint:
        sft_params = sft_checkpoint['params']
    else:
        sft_params = sft_checkpoint
    
    print("SFT model loaded successfully")
    print(f"SFT params keys: {list(sft_params.keys())}")
    
except FileNotFoundError:
    print("SFT model not found! Initializing from scratch...")
    sft_params = None

Loading SFT model from /kaggle/working/step2_sft_model/sft_best.pkl...
SFT model loaded successfully
SFT params keys: ['Dense_0', 'Embed_0', 'RMSNorm_0', 'block_0', 'block_1', 'block_2', 'block_3', 'block_4', 'block_5']


In [62]:
def train_reward_model(sft_params, train_data, val_data, target_epochs=3):
    num_devices = jax.device_count()
    print(f"\n{'='*100}")
    print(f"TRAINING REWARD MODEL ON {num_devices} DEVICES")
    print(f"{'='*100}")
    print(f"Train: {len(train_data):,} | Val: {len(val_data):,}")
    
    rng = jax.random.PRNGKey(0)
    rng, init_rng = jax.random.split(rng)
    
    steps_per_epoch = len(train_data) // BATCH_SIZE
    total_steps = steps_per_epoch * target_epochs
    
    print(f"Steps/epoch: {steps_per_epoch:,} | Total: {total_steps:,}")
    print(f"{'='*100}\n")
    
    rm_state = replicate(create_rm_train_state(sft_params, init_rng, LEARNING_RATE, total_steps))
    
    history = []
    best_val_acc = 0.0
    best_params = None
    start_time = time.time()
    
    for epoch in range(target_epochs):
        epoch_start = time.time()
        jax.clear_caches()
        gc.collect()
        
        train_metrics = {
            'loss': [], 'accuracy': [], 'margin': [],
            'mean_chosen': [], 'mean_rejected': [],
            'std_chosen': [], 'std_rejected': [],
            'confidence': [], 'cross_entropy': []
        }
        
        pbar = tqdm(
            total=steps_per_epoch,
            desc=f"Epoch {epoch+1}/{target_epochs}",
            unit="step",
            leave=True
        )
        
        # ================== TRAIN LOOP ==================
        for batch in create_rm_batches(train_data, BATCH_SIZE, num_devices, shuffle=True):
            rm_state, loss, metrics = rm_train_step(rm_state, batch)
            
            loss = jax.block_until_ready(loss)
            metrics = jax.block_until_ready(metrics)
            
            train_metrics['loss'].append(float(loss[0]))
            for key in metrics.keys():
                train_metrics[key].append(float(metrics[key][0]))
            
            pbar.update(1)
            pbar.set_postfix({
                'loss': f"{train_metrics['loss'][-1]:.4f}",
                'acc': f"{train_metrics['accuracy'][-1]:.3f}",
                'margin': f"{train_metrics['margin'][-1]:.3f}"
            })
        
        pbar.close()
        train_avg = {k: np.mean(v) for k, v in train_metrics.items()}
        
        # ================== VALIDATION ==================
        print("\nValidating...")
        val_metrics = {
            'loss': [], 'accuracy': [], 'margin': [],
            'mean_chosen': [], 'mean_rejected': [],
            'std_chosen': [], 'std_rejected': [],
            'confidence': [], 'cross_entropy': []
        }
        
        for val_batch in create_rm_batches(val_data, BATCH_SIZE, num_devices, shuffle=False):
            metrics = rm_val_step(rm_state.params, val_batch)
            metrics = jax.block_until_ready(metrics)
            
            for key in metrics.keys():
                val_metrics[key].append(float(metrics[key][0]))
        
        val_avg = {k: np.mean(v) for k, v in val_metrics.items()}
        
        # ================== LOGGING ==================
        epoch_time = time.time() - epoch_start
        total_time = time.time() - start_time
        
        log_entry = {
            'Epoch': f'{epoch+1}/{target_epochs}',
            'Train Loss': f"{train_avg['loss']:.4f}",
            'Train Acc': f"{train_avg['accuracy']:.3f}",
            'Train Margin': f"{train_avg['margin']:.3f}",
            'Val Loss': f"{val_avg['loss']:.4f}",
            'Val Acc': f"{val_avg['accuracy']:.3f}",
            'Val Margin': f"{val_avg['margin']:.3f}",
            'Confidence': f"{val_avg['confidence']:.3f}",
            'Epoch Time': f'{epoch_time/60:.1f}min',
            'Total Time': f'{total_time/60:.1f}min'
        }
        
        history.append(log_entry)
        print("\n" + tabulate([log_entry], headers="keys", tablefmt="simple"))
        
        # ================== BEST MODEL TRACKING ==================
        if val_avg['accuracy'] > best_val_acc:
            best_val_acc = val_avg['accuracy']
            print(f"✓ New best val accuracy: {best_val_acc:.3f}")
            
            rm_state_save = unreplicate(rm_state)
            best_params = rm_state_save.params
            
            with open("/kaggle/working/best_reward_model_3.pkl", 'wb') as f:
                pickle.dump({'params': best_params, 'val_acc': best_val_acc}, f)
        
        print(f"{'='*100}\n")
    
    # ================== TRAINING COMPLETE ==================
    print(f"\n{'='*100}")
    print("REWARD MODEL TRAINING COMPLETE")
    print(f"{'='*100}")
    print(f"Best val accuracy: {best_val_acc:.3f}")
    print(tabulate(history, headers="keys", tablefmt="grid"))
    print(f"{'='*100}")
    
    rm_state = unreplicate(rm_state)
    
    # Safety fallback
    if best_params is None:
        print("⚠ No validation improvement. Using final model as best model.")
        best_params = rm_state.params
    
    with open("/kaggle/working/final_reward_model_3.pkl", 'wb') as f:
        pickle.dump({'params': rm_state.params, 'history': history}, f)
    
    print("\nSaved: /kaggle/working/final_reward_model_3.pkl")
    print("Saved: /kaggle/working/best_reward_model_3.pkl")
    
    return rm_state, best_params, history

In [63]:
final_rm_state_3, best_params_3, history_3 = train_reward_model(sft_params,train_data_3,val_data_3,target_epochs=1)


TRAINING REWARD MODEL ON 8 DEVICES
Train: 900 | Val: 100
Steps/epoch: 112 | Total: 112

Loading SFT weights into reward model...
  Loaded Embed_0
  Loaded block_0
  Loaded block_1
  Loaded block_2
  Loaded block_3
  Loaded block_4
  Loaded block_5
  Loaded RMSNorm_0
  Skipped Dense_0 (reward head)


Epoch 1/1: 100%|██████████| 112/112 [12:16<00:00,  6.57s/step, loss=0.8892, acc=0.750, margin=1.706]   



Validating...

Epoch      Train Loss    Train Acc    Train Margin    Val Loss    Val Acc    Val Margin    Confidence  Epoch Time    Total Time
-------  ------------  -----------  --------------  ----------  ---------  ------------  ------------  ------------  ------------
1/1            0.5337        0.752           1.164      0.5367       0.74         1.303         0.699  12.4min       12.4min
✓ New best val accuracy: 0.740


REWARD MODEL TRAINING COMPLETE
Best val accuracy: 0.740
+---------+--------------+-------------+----------------+------------+-----------+--------------+--------------+--------------+--------------+
| Epoch   |   Train Loss |   Train Acc |   Train Margin |   Val Loss |   Val Acc |   Val Margin |   Confidence | Epoch Time   | Total Time   |
+=========+==============+=============+================+============+===========+==============+==============+==============+==============+
| 1/1     |       0.5337 |       0.752 |          1.164 |     0.5367 |      0.74 | 

In [64]:
print("=" * 80)
print("REWARD MODEL 3 EVALUATION")
print("=" * 80)

with open("/kaggle/working/best_reward_model_3.pkl", 'rb') as f:
    checkpoint = pickle.load(f)

eval_params  = checkpoint['params']
best_val_acc = checkpoint['val_acc']

embed_shape = eval_params['Embed_0']['embedding'].shape
if embed_shape[0] != 50257:
    eval_params = jax.tree_util.tree_map(lambda x: x[0] if x.ndim >= 2 else x, eval_params)

eval_params_replicated = jax.device_put_replicated(eval_params, jax.local_devices())
print(f"Loaded | Best Val Acc: {best_val_acc:.4f}")

def _rm_val_step_fn_2(params, batch):
    rm               = RewardModel(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    chosen_rewards   = rm.apply({'params': params}, batch['chosen_input_ids'],   train=False)
    rejected_rewards = rm.apply({'params': params}, batch['rejected_input_ids'], train=False)
    metrics          = compute_reward_metrics(chosen_rewards, rejected_rewards)
    metrics['loss']  = bradley_terry_loss(chosen_rewards, rejected_rewards)
    return jax.tree.map(lambda x: jax.lax.pmean(x, axis_name='data'), metrics)

_rm_val_step_2 = jax.pmap(_rm_val_step_fn_2, axis_name='data')

all_chosen, all_rejected, all_margins, all_conf = [], [], [], []

for val_batch in tqdm(create_rm_batches(val_data_3, BATCH_SIZE, jax.device_count(), shuffle=False), desc="Evaluating"):
    metrics = jax.block_until_ready(_rm_val_step_2(eval_params_replicated, val_batch))
    all_chosen.append(float(np.array(metrics['mean_chosen'])[0]))
    all_rejected.append(float(np.array(metrics['mean_rejected'])[0]))
    all_margins.append(float(np.array(metrics['margin'])[0]))
    all_conf.append(float(np.array(metrics['confidence'])[0]))

chosen_arr   = np.array(all_chosen)
rejected_arr = np.array(all_rejected)
margins_arr  = np.array(all_margins)
conf_arr     = np.array(all_conf)

pred_labels = (chosen_arr > rejected_arr).astype(int)
scores      = chosen_arr - rejected_arr

accuracy    = np.mean(pred_labels)
mean_margin = np.mean(margins_arr)
mean_conf   = np.mean(conf_arr)
separation  = (np.mean(chosen_arr) - np.mean(rejected_arr)) / (np.std(chosen_arr) + np.std(rejected_arr) + 1e-8)
auc_roc     = roc_auc_score(pred_labels, scores) if len(np.unique(pred_labels)) > 1 else float('nan')

print("\n" + tabulate([
    ["Accuracy",         f"{accuracy:.4f}"],
    ["AUC-ROC",          f"{auc_roc:.4f}" if not np.isnan(auc_roc) else "N/A (all correct)"],
    ["Mean Margin",      f"{mean_margin:.4f}"],
    ["Mean Confidence",  f"{mean_conf:.4f}"],
    ["Mean Chosen",      f"{np.mean(chosen_arr):.4f}"],
    ["Mean Rejected",    f"{np.mean(rejected_arr):.4f}"],
    ["Separation Ratio", f"{separation:.4f}"],
], headers=["Metric", "Value"], tablefmt="grid"))

print("\n" + "=" * 80)
grade = "Excellent" if accuracy >= 0.85 else "Good" if accuracy >= 0.75 else "Moderate" if accuracy >= 0.65 else "Poor"
print(f"  Accuracy : {accuracy:.4f}  ->  {grade}")
print(f"  AUC-ROC  : {auc_roc:.4f}" if not np.isnan(auc_roc) else "  AUC-ROC  : N/A (model predicts all correct, no negatives)")
print(f"  Margin   : {mean_margin:.4f}")
print(f"\n  Model is {'READY' if accuracy >= 0.70 else 'NOT YET READY'} for GRPO stage.")
print("=" * 80)

REWARD MODEL 3 EVALUATION
Loaded | Best Val Acc: 0.7396


Evaluating: 12it [00:05,  2.35it/s]


+------------------+-------------------+
| Metric           | Value             |
+==================+===================+
| Accuracy         | 1.0000            |
+------------------+-------------------+
| AUC-ROC          | N/A (all correct) |
+------------------+-------------------+
| Mean Margin      | 1.3026            |
+------------------+-------------------+
| Mean Confidence  | 0.6986            |
+------------------+-------------------+
| Mean Chosen      | -0.9884           |
+------------------+-------------------+
| Mean Rejected    | -2.2910           |
+------------------+-------------------+
| Separation Ratio | 0.8761            |
+------------------+-------------------+

  Accuracy : 1.0000  ->  Excellent
  AUC-ROC  : N/A (model predicts all correct, no negatives)
  Margin   : 1.3026

  Model is READY for GRPO stage.


In [65]:
OUTPUT_DIR = "/kaggle/working/reward_model_3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Saving reward models...")

final_checkpoint = {
    'params': final_rm_state_3.params,
    'model_config': {
        'vocab_size': VOCAB_SIZE,
        'num_layers': NUM_LAYERS,
        'num_heads': NUM_HEADS,
        'head_dim': HEAD_DIM,
        'mlp_dim': MLP_DIM,
        'd_model': D_MODEL,
        'max_seq_len': MAX_SEQ_LEN,
    },
    'training_history': history_3
}

with open(f"{OUTPUT_DIR}/final_reward_model.pkl", 'wb') as f:
    pickle.dump(final_checkpoint, f)

print(f"Final model saved to {OUTPUT_DIR}/final_reward_model.pkl")

best_checkpoint = {'params': best_params_3,'model_config': final_checkpoint['model_config'],'training_history': history_3}

with open(f"{OUTPUT_DIR}/best_reward_model.pkl", 'wb') as f:
    pickle.dump(best_checkpoint, f)

print(f"Best model saved to {OUTPUT_DIR}/best_reward_model.pkl")

import json
with open(f"{OUTPUT_DIR}/training_history.json", 'w') as f:
    json.dump(history_3, f, indent=2)

print(f"Training history saved to {OUTPUT_DIR}/training_history.json")

print("\nSaved files:")
print(f"1. {OUTPUT_DIR}/final_reward_model.pkl")
print(f"2. {OUTPUT_DIR}/best_reward_model.pkl")
print(f"3. {OUTPUT_DIR}/training_history.json")

Saving reward models...
Final model saved to /kaggle/working/reward_model_3/final_reward_model.pkl
Best model saved to /kaggle/working/reward_model_3/best_reward_model.pkl
Training history saved to /kaggle/working/reward_model_3/training_history.json

Saved files:
1. /kaggle/working/reward_model_3/final_reward_model.pkl
2. /kaggle/working/reward_model_3/best_reward_model.pkl
3. /kaggle/working/reward_model_3/training_history.json


# Step 5: Reinforcement Learning Fine-Tuning (GRPO) 

In [66]:
NUM_DEVICES = jax.device_count()
VOCAB_SIZE   = 50257
MAX_SEQ_LEN  = 48
D_MODEL      = 768
NUM_LAYERS   = 6
NUM_HEADS    = 12
HEAD_DIM     = 64
MLP_DIM      = D_MODEL * 4
DROPOUT      = 0.1

GRPO_LR          = 1e-6
GRPO_EPOCHS      = 1
GRPO_G           = 2
GRPO_CLIP_EPS    = 0.2
GRPO_KL_BETA     = 0.01
GRPO_MAX_NEW     = 16
GRPO_BATCH_SIZE  = NUM_DEVICES
GRPO_GRAD_CLIP   = 1.0
GRPO_WARMUP      = 50
GRPO_TEMPERATURE = 0.9

print(f"Devices: {NUM_DEVICES}")
print(f"G={GRPO_G} | clip={GRPO_CLIP_EPS} | kl_beta={GRPO_KL_BETA} | lr={GRPO_LR}")

Devices: 8
G=2 | clip=0.2 | kl_beta=0.01 | lr=1e-06


In [67]:
RM_REWARD_MODEL_1="/kaggle/working/reward_model_1/best_reward_model.pkl"
RM_REWARD_MODEL_2="/kaggle/working/reward_model_2/best_reward_model.pkl"
RM_REWARD_MODEL_3="/kaggle/working/reward_model_3/best_reward_model.pkl"
POLICY_MODEL="/kaggle/working/step2_sft_model/sft_best.pkl"
REFERENCE_MODEL="/kaggle/working/step2_sft_model/sft_best.pkl"

In [68]:
with open(POLICY_MODEL, 'rb') as f:
    policy_params = pickle.load(f)['params']

with open(REFERENCE_MODEL, 'rb') as f:
    ref_params = pickle.load(f)['params']
ref_params = jax.device_put_replicated(ref_params, jax.local_devices())

def load_rm(path):
    with open(path, 'rb') as f:
        ckpt = pickle.load(f)
    params = ckpt['params']
    leaves = jax.tree_util.tree_leaves(params)
    if leaves[0].ndim >= 1 and leaves[0].shape[0] == NUM_DEVICES:
        params = jax.tree_util.tree_map(lambda x: x[0], params)
    return jax.device_put_replicated(params, jax.local_devices())

rm_params_1 = load_rm(RM_REWARD_MODEL_1)
rm_params_2 = load_rm(RM_REWARD_MODEL_2)
rm_params_3 = load_rm(RM_REWARD_MODEL_3)

print(f"Policy:     {sum(x.size for x in jax.tree_util.tree_leaves(policy_params))/1e6:.1f}M params")
print(f"Reference:  loaded and replicated")
print(f"RM 1/2/3:   loaded and replicated")

Policy:     133.8M params
Reference:  loaded and replicated
RM 1/2/3:   loaded and replicated


In [69]:
tokenizer = tiktoken.get_encoding("gpt2")
EOT       = tokenizer.eot_token

df = pd.read_csv("/kaggle/input/mini-llm-code/grpo_dataset.csv", nrows=500)
df = df.rename(columns={"content": "prompt"})
df = df[['prompt']].dropna()
df = df[df['prompt'].str.len() > 30].reset_index(drop=True)

prompts_encoded = []
for q in df['prompt'].values:
    prompt = f"### Instruction:\n{q.strip()}\n\n### Response:\n"
    tokens = tokenizer.encode(prompt, allowed_special="all")[:MAX_SEQ_LEN // 2]
    prompts_encoded.append(tokens)

print(f"Total prompts: {len(prompts_encoded)}")
print(f"Sample tokens: {prompts_encoded[0][:10]}...")

Total prompts: 500
Sample tokens: [21017, 46486, 25, 198, 15056, 281, 7177, 286, 37014, 4600]...


In [70]:
def get_reward(rm_params, tokens_np):
    tokens = jnp.array(tokens_np)[:MAX_SEQ_LEN]
    pad    = MAX_SEQ_LEN - tokens.shape[0]
    if pad > 0:
        tokens = jnp.pad(tokens, (0, pad), constant_values=EOT)
    tokens = jnp.broadcast_to(tokens, (NUM_DEVICES, MAX_SEQ_LEN))
    reward = jax.pmap(lambda p, t: RewardModel(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN).apply({'params': p}, t[None, :], train=False), axis_name='data')(rm_params, tokens)
    return float(reward[0][0])

def ensemble_reward(tokens_np):
    r1 = get_reward(rm_params_1, tokens_np)
    r2 = get_reward(rm_params_2, tokens_np)
    r3 = get_reward(rm_params_3, tokens_np)
    return (r1 + r2 + r3) / 3.0

In [71]:
def generate_completion(params, prompt_tokens, temperature=GRPO_TEMPERATURE, max_new=GRPO_MAX_NEW, seed=0):
    model  = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    rng    = jax.random.PRNGKey(seed)
    tokens = list(prompt_tokens)

    for _ in range(max_new):
        inp     = jnp.array([tokens[-MAX_SEQ_LEN:]], dtype=jnp.int32)
        logits  = model.apply({'params': params}, inp, train=False)
        next_lp = logits[0, -1, :] / temperature
        rng, sk = jax.random.split(rng)
        next_id = int(jax.random.categorical(sk, next_lp))
        tokens.append(next_id)
        if next_id == EOT:
            break

    return tokens

def generate_group(params, prompt_tokens):
    return [generate_completion(params, prompt_tokens, seed=g * 997) for g in range(GRPO_G)]

In [72]:
def compute_seq_log_prob(params, tokens_arr):
    model      = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    inp        = jnp.array([tokens_arr])
    logits     = model.apply({'params': params}, inp, train=False)
    log_probs  = jax.nn.log_softmax(logits[0, :-1, :], axis=-1)
    targets    = jnp.array(tokens_arr[1:])
    token_lp   = jnp.take_along_axis(log_probs, targets[:, None], axis=-1).squeeze(-1)
    return float(token_lp.mean())

In [73]:
def grpo_loss_fn(policy_params, ref_params, batch, advantages, old_log_probs, dropout_rng):
    model             = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    tokens            = batch['tokens']
    loss_mask         = batch['loss_mask']

    @jax.checkpoint
    def policy_forward(params):
        return model.apply({'params': params}, tokens, train=True, rngs={'dropout': dropout_rng})

    @jax.checkpoint
    def ref_forward(params):
        return model.apply({'params': params}, tokens, train=False)

    policy_logits     = policy_forward(policy_params)
    ref_logits        = jax.lax.stop_gradient(ref_forward(ref_params))
    policy_lp         = jax.nn.log_softmax(policy_logits[:, :-1, :], axis=-1)
    targets           = tokens[:, 1:]
    token_lp          = jnp.take_along_axis(policy_lp, targets[:, :, None], axis=-1).squeeze(-1)
    seq_lp            = (token_lp * loss_mask[:, 1:]).sum(-1) / jnp.maximum(loss_mask[:, 1:].sum(-1), 1.0)
    ratio             = jnp.exp(seq_lp - old_log_probs)
    clipped           = jnp.clip(ratio, 1 - GRPO_CLIP_EPS, 1 + GRPO_CLIP_EPS)
    policy_loss       = -jnp.mean(jnp.minimum(ratio * advantages, clipped * advantages))
    ref_lp            = jax.nn.log_softmax(ref_logits[:, :-1, :], axis=-1)
    kl                = jnp.sum(jnp.exp(policy_lp) * (policy_lp - ref_lp), axis=-1)
    kl_loss           = jnp.mean(kl)
    total_loss        = policy_loss + GRPO_KL_BETA * kl_loss
    
    return total_loss, {'policy_loss': policy_loss,'kl_loss': kl_loss, 'total_loss':  total_loss}

In [74]:
class GRPOTrainState(train_state.TrainState):
    dropout_rng: jax.random.PRNGKey

def create_grpo_state(policy_params, rng, total_steps):
    warmup   = min(GRPO_WARMUP, max(1, total_steps // 10))
    schedule = optax.warmup_cosine_decay_schedule(init_value=0.0,peak_value=GRPO_LR,warmup_steps=warmup,decay_steps=max(total_steps, warmup + 1),end_value=GRPO_LR * 0.1)
    tx = optax.chain(optax.clip_by_global_norm(GRPO_GRAD_CLIP),optax.adamw(schedule, weight_decay=0.01))
    model = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    return GRPOTrainState.create(apply_fn=model.apply, params=policy_params, tx=tx, dropout_rng=rng)

@partial(jax.pmap, axis_name='data', donate_argnums=(0,))
def grpo_train_step(state, ref_params, batch, advantages, old_log_probs):
    dropout_rng, new_rng = jax.random.split(state.dropout_rng)
    (loss, metrics), grads = jax.value_and_grad(lambda p: grpo_loss_fn(p, ref_params, batch, advantages, old_log_probs, dropout_rng),has_aux=True)(state.params)
    grads   = jax.lax.pmean(grads,   axis_name='data')
    loss    = jax.lax.pmean(loss,    axis_name='data')
    metrics = jax.tree.map(lambda x: jax.lax.pmean(x, axis_name='data'), metrics)
    state   = state.apply_gradients(grads=grads).replace(dropout_rng=new_rng)
    return state, loss, metrics

print("GRPO train step defined")

GRPO train step defined


In [75]:
def train_grpo(policy_params, ref_params, prompts, target_epochs=GRPO_EPOCHS):
    num_devices     = jax.device_count()
    rng             = jax.random.PRNGKey(42)
    steps_per_epoch = len(prompts) // GRPO_BATCH_SIZE
    total_steps     = steps_per_epoch * target_epochs

    state = replicate(create_grpo_state(policy_params, rng, total_steps))

    print("=" * 100)
    print(f"GRPO TRAINING | Devices={num_devices} | Prompts={len(prompts)} | G={GRPO_G} | Epochs={target_epochs}")
    print(f"Steps/epoch={steps_per_epoch} | Total={total_steps}")
    print("=" * 100)

    history      = []
    best_reward  = -float('inf')
    best_params  = None
    start_time   = time.time()

    for epoch in range(target_epochs):
        epoch_start   = time.time()
        epoch_metrics = {'total_loss': [], 'policy_loss': [], 'kl_loss': [], 'mean_reward': [], 'best_reward': []}

        pbar = tqdm(range(steps_per_epoch), desc=f"Epoch {epoch+1}/{target_epochs}", unit="step", leave=True)
        np.random.shuffle(prompts)

        for step in pbar:
            if step % 10 == 0:
                jax.clear_caches()

            batch_prompts  = prompts[step * GRPO_BATCH_SIZE:(step + 1) * GRPO_BATCH_SIZE]
            current_params = jax.tree_util.tree_map(lambda x: np.array(x[0]), state.params)

            all_tokens, all_masks, all_advantages, all_old_lp, all_rewards = [], [], [], [], []

            for prompt_tokens in batch_prompts:
                completions = generate_group(current_params, prompt_tokens)

                rewards = [ensemble_reward(np.pad(np.array(c[:MAX_SEQ_LEN], dtype=np.int32), (0, max(0, MAX_SEQ_LEN - len(c[:MAX_SEQ_LEN]))),
                           constant_values=EOT)) for c in completions]

                rewards_arr = np.array(rewards)
                advantages  = (rewards_arr - rewards_arr.mean()) / (rewards_arr.std() + 1e-8)
                best_idx    = int(np.argmax(rewards))
                best_comp   = completions[best_idx]
                best_adv    = float(advantages[best_idx])

                prompt_len  = len(prompt_tokens)
                comp_arr    = np.array(best_comp[:MAX_SEQ_LEN], dtype=np.int32)
                mask        = np.zeros(MAX_SEQ_LEN, dtype=np.float32)
                resp_end    = min(len(best_comp), MAX_SEQ_LEN)
                mask[prompt_len:resp_end] = 1.0
                pad         = MAX_SEQ_LEN - len(comp_arr)
                if pad > 0:
                    comp_arr = np.pad(comp_arr, (0, pad), constant_values=EOT)

                old_lp = compute_seq_log_prob(current_params, comp_arr)

                all_tokens.append(comp_arr)
                all_masks.append(mask)
                all_advantages.append(best_adv)
                all_old_lp.append(old_lp)
                all_rewards.append(float(rewards_arr.mean()))

            tokens_np     = np.stack(all_tokens).reshape(num_devices, -1, MAX_SEQ_LEN)
            masks_np      = np.stack(all_masks).reshape(num_devices, -1, MAX_SEQ_LEN)
            advantages_np = np.array(all_advantages, dtype=np.float32).reshape(num_devices, -1)
            old_lp_np     = np.array(all_old_lp,    dtype=np.float32).reshape(num_devices, -1)

            batch = {'tokens':    jnp.array(tokens_np,  dtype=jnp.int32),'loss_mask': jnp.array(masks_np,   dtype=jnp.float32)}

            state, loss, metrics = grpo_train_step(state, ref_params,batch,jnp.array(advantages_np),jnp.array(old_lp_np))

            loss    = jax.block_until_ready(loss)
            metrics = jax.block_until_ready(metrics)

            epoch_metrics['total_loss'].append(float(loss[0]))
            epoch_metrics['policy_loss'].append(float(metrics['policy_loss'][0]))
            epoch_metrics['kl_loss'].append(float(metrics['kl_loss'][0]))
            epoch_metrics['mean_reward'].append(np.mean(all_rewards))
            epoch_metrics['best_reward'].append(max(all_rewards))

            pbar.set_postfix({'loss':f"{epoch_metrics['total_loss'][-1]:.4f}",'reward':f"{epoch_metrics['mean_reward'][-1]:.3f}",'best_reward':f"{epoch_metrics['best_reward'][-1]:.3f}",
                'kl':f"{epoch_metrics['kl_loss'][-1]:.4f}",
            })

        avg        = {k: float(np.mean(v)) for k, v in epoch_metrics.items()}
        epoch_time = (time.time() - epoch_start) / 60
        total_time = (time.time() - start_time)  / 60

        log_entry = {
            'Epoch':       f"{epoch+1}/{target_epochs}",
            'Total Loss':  f"{avg['total_loss']:.4f}",
            'Policy Loss': f"{avg['policy_loss']:.4f}",
            'KL Loss':     f"{avg['kl_loss']:.4f}",
            'Mean Reward': f"{avg['mean_reward']:.4f}",
            'Best Reward': f"{avg['best_reward']:.4f}",
            'Epoch Time':  f"{epoch_time:.1f}min",
            'Total Time':  f"{total_time:.1f}min",
        }

        history.append(log_entry)
        print("\n" + tabulate([log_entry], headers="keys", tablefmt="heavy_grid"))

        if avg['mean_reward'] > best_reward:
            best_reward  = avg['mean_reward']
            best_params  = jax.tree_util.tree_map(lambda x: np.array(x[0]), state.params)
            with open("/kaggle/working/grpo_best.pkl", 'wb') as f:
                pickle.dump({'params': best_params, 'reward': best_reward, 'history': history}, f)
            print(f"Saved best model — reward={best_reward:.4f}")

        jax.clear_caches()
        gc.collect()
        print("=" * 100)

    print("\nGRPO TRAINING COMPLETE")
    print(f"Best mean reward: {best_reward:.4f}")
    print(tabulate(history, headers="keys", tablefmt="grid"))

    final_params = jax.tree_util.tree_map(lambda x: np.array(x[0]), state.params)
    return final_params, best_params, history

In [76]:
grpo_state, grpo_best_params, grpo_history = train_grpo(policy_params = policy_params, ref_params = ref_params,prompts = prompts_encoded,target_epochs = GRPO_EPOCHS)

GRPO TRAINING | Devices=8 | Prompts=500 | G=2 | Epochs=1
Steps/epoch=62 | Total=62


Epoch 1/1: 100%|██████████| 62/62 [3:28:37<00:00, 201.89s/step, loss=-1.1997, reward=2.191, best_reward=2.798, kl=0.0346]  



┏━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Epoch   ┃   Total Loss ┃   Policy Loss ┃   KL Loss ┃   Mean Reward ┃   Best Reward ┃ Epoch Time   ┃ Total Time   ┃
┣━━━━━━━━━╋━━━━━━━━━━━━━━╋━━━━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━━━━━━━╋━━━━━━━━━━━━━━━╋━━━━━━━━━━━━━━╋━━━━━━━━━━━━━━┫
┃ 1/1     ┃      -1.1997 ┃          -1.2 ┃    0.0328 ┃        2.0978 ┃        2.9704 ┃ 208.6min     ┃ 208.6min     ┃
┗━━━━━━━━━┻━━━━━━━━━━━━━━┻━━━━━━━━━━━━━━━┻━━━━━━━━━━━┻━━━━━━━━━━━━━━━┻━━━━━━━━━━━━━━━┻━━━━━━━━━━━━━━┻━━━━━━━━━━━━━━┛
Saved best model — reward=2.0978

GRPO TRAINING COMPLETE
Best mean reward: 2.0978
+---------+--------------+---------------+-----------+---------------+---------------+--------------+--------------+
| Epoch   |   Total Loss |   Policy Loss |   KL Loss |   Mean Reward |   Best Reward | Epoch Time   | Total Time   |
+=========+==============+===============+===========+===============+===============+============

In [77]:
OUTPUT_DIR = "/kaggle/working/step5_grpo_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

final_params = grpo_state

model_config = {'vocab_size':  VOCAB_SIZE,'num_layers':  NUM_LAYERS,'num_heads':   NUM_HEADS, 'head_dim':    HEAD_DIM, 'mlp_dim':MLP_DIM,
    'd_model':     D_MODEL,
    'max_seq_len': MAX_SEQ_LEN,
}

with open(f"{OUTPUT_DIR}/grpo_final.pkl", 'wb') as f:
    pickle.dump({'params': final_params, 'model_config': model_config, 'history': grpo_history}, f)

with open(f"{OUTPUT_DIR}/grpo_best.pkl", 'wb') as f:
    pickle.dump({'params': grpo_best_params, 'model_config': model_config, 'history': grpo_history}, f)

total_params = sum(x.size for x in jax.tree_util.tree_leaves(final_params))
print(f"Saved {total_params/1e6:.1f}M params")
print(f"Final : {OUTPUT_DIR}/grpo_final.pkl")
print(f"Best  : {OUTPUT_DIR}/grpo_best.pkl")

Saved 133.8M params
Final : /kaggle/working/step5_grpo_model/grpo_final.pkl
Best  : /kaggle/working/step5_grpo_model/grpo_best.pkl


In [78]:
print("=" * 80)
print("GRPO TRAINED MODEL - INTERACTIVE INFERENCE")
print("=" * 80)

with open("/kaggle/working/step5_grpo_model/grpo_best.pkl", 'rb') as f:
    ckpt = pickle.load(f)
inference_params = ckpt['params']
print(f"Loaded GRPO best model")

def generate_answer(params, query, max_new=200, temperature=0.7, top_k=40):
    prompt    = f"### Instruction:\n{query.strip()}\n\n### Response:\n"
    input_ids = tokenizer.encode(prompt, allowed_special="all")
    tokens    = jnp.array([input_ids], dtype=jnp.int32)
    
    model     = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    rng       = jax.random.PRNGKey(42)
    generated = []

    caches = [{'k': jnp.zeros((1, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM)),
               'v': jnp.zeros((1, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM))}
              for _ in range(NUM_LAYERS)]

    logits, caches = model.apply({'params': params}, tokens, train=False,
                                  caches=caches, cache_pos=0)
    cache_pos = len(input_ids)

    for _ in range(max_new):
        next_logits         = logits[0, -1, :] / temperature
        topk_vals, topk_idx = jax.lax.top_k(next_logits, top_k)
        probs               = jax.nn.softmax(topk_vals)
        rng, sk             = jax.random.split(rng)
        chosen              = topk_idx[jax.random.categorical(sk, jnp.log(probs + 1e-10))]
        chosen_int          = int(chosen)

        if chosen_int == EOT:
            break

        generated.append(chosen_int)

        next_tok = jnp.array([[chosen_int]], dtype=jnp.int32)
        logits, caches = model.apply({'params': params}, next_tok, train=False,
                                     caches=caches, cache_pos=cache_pos)
        cache_pos += 1

    return tokenizer.decode(generated)

questions = [
    "Write a Python function to check if a number is prime.",
    "Write a Python function to reverse a linked list.",
    "Write a Python function to find the maximum subarray sum using Kadane's algorithm."
]

for i, question in enumerate(tqdm(questions, desc="Processing questions", unit="q"), 1):
    print(f"\n{'='*80}")
    print(f"Question {i}: {question}")
    print(f"{'='*80}")

    answer = generate_answer(inference_params, question,
                             max_new=200,
                             temperature=0.7,
                             top_k=40)

    print(f"Answer:\n{answer.strip()}")
    print(f"{'='*80}")

GRPO TRAINED MODEL - INTERACTIVE INFERENCE
Loaded GRPO best model


Processing questions:   0%|          | 0/3 [00:00<?, ?q/s]


Question 1: Write a Python function to check if a number is prime.


Processing questions:  33%|███▎      | 1/3 [01:13<02:26, 73.26s/q]

Answer:
..

#__
"""
         return

	:





 -

                      ` the-
#:
                              """                          is a for-






          # is

Question 2: Write a Python function to reverse a linked list.


Processing questions:  67%|██████▋   | 2/3 [02:05<01:00, 60.74s/q]

Answer:
.


#


 =

def_:



from import
	 =





 #

       
           :
  to


#    
##

#


  """      









def(s
 







 





_



:501









          """    
:-:


 :

#


     








      :




  
#:

Question 3: Write a Python function to find the maximum subarray sum using Kadane's algorithm.


Processing questions: 100%|██████████| 3/3 [02:58<00:00, 59.58s/q]

Answer:
.:


#


class.
def_
     # in the-.





 # the to a of




            :
  to


#    
#/

#


  """                 -`,
          





from.


: a









          """    

         :


                    :




  
## in


# Full Agentic Code Using Own Model

In [86]:
def execute_code(code, timeout=10):
    try:
        old_stdout = sys.stdout
        sys.stdout  = io.StringIO()
        exec_globals = {}
        exec(code, exec_globals)
        output = sys.stdout.getvalue()
        sys.stdout = old_stdout
        return {'status': 'success', 'output': output if output else 'Code ran successfully'}
    except Exception as e:
        sys.stdout = old_stdout
        return {'status': 'error', 'output': str(e)}

def classify_question(query):
    query_lower = query.lower()
    if any(k in query_lower for k in ['sort', 'search', 'array', 'list', 'tree', 'graph', 'dynamic', 'prime', 'fibonacci', 'factorial']):
        return 'algorithm'
    elif any(k in query_lower for k in ['class', 'object', 'inherit', 'polymorphism']):
        return 'oop'
    elif any(k in query_lower for k in ['file', 'read', 'write', 'csv', 'json']):
        return 'file_io'
    elif any(k in query_lower for k in ['api', 'request', 'http', 'web']):
        return 'web'
    elif any(k in query_lower for k in ['sql', 'database', 'query']):
        return 'database'
    else:
        return 'general'

def extract_code_block(text):
    import re
    patterns = [
        r'```python\n(.*?)```',
        r'```\n(.*?)```',
        r'def .*?(?=\n\n|\Z)',
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            return match.group(1) if '```' in pattern else match.group(0)
    return None

print("Agent tools defined")

Agent tools defined


In [87]:
AGENT_PROMPTS = {
    'algorithm': "### Instruction:\n{query}\n\n### Response:\n",
    'oop':       "### Instruction:\n{query}\n\n### Response:\n",
    'file_io':   "### Instruction:\n{query}\n\n### Response:\n",
    'general':   "### Instruction:\n{query}\n\n### Response:\n",
    'web':       "### Instruction:\n{query}\n\n### Response:\n",
    'database':  "### Instruction:\n{query}\n\n### Response:\n",
}

def build_prompt(query, question_type, error=None, prev_code=None):
    base = AGENT_PROMPTS.get(question_type, AGENT_PROMPTS['general'])
    return base.format(query=query.strip())

print("Agent prompts defined")

Agent prompts defined


In [90]:
def agent_generate(params, prompt, max_new=200, temperature=0.7, top_k=40):
    input_ids = tokenizer.encode(prompt, allowed_special="all")

    max_prompt_tokens = MAX_SEQ_LEN - 1
    if len(input_ids) > max_prompt_tokens:
        input_ids = input_ids[-max_prompt_tokens:]

    tokens    = jnp.array([input_ids], dtype=jnp.int32)
    model     = Transformer(VOCAB_SIZE, NUM_LAYERS, NUM_HEADS, HEAD_DIM, MLP_DIM, D_MODEL, MAX_SEQ_LEN)
    rng       = jax.random.PRNGKey(42)
    generated = []

    caches = [{'k': jnp.zeros((1, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM)),
               'v': jnp.zeros((1, NUM_HEADS, MAX_SEQ_LEN, HEAD_DIM))}
              for _ in range(NUM_LAYERS)]

    logits, caches = model.apply({'params': params}, tokens, train=False,
                                  caches=caches, cache_pos=0)
    cache_pos = len(input_ids)

    pbar = tqdm(range(max_new), desc="Generating", leave=False)
    for _ in pbar:
        if cache_pos >= MAX_SEQ_LEN:
            pbar.close()
            break
        next_logits         = logits[0, -1, :] / temperature
        topk_vals, topk_idx = jax.lax.top_k(next_logits, top_k)
        probs               = jax.nn.softmax(topk_vals)
        rng, sk             = jax.random.split(rng)
        chosen              = topk_idx[jax.random.categorical(sk, jnp.log(probs + 1e-10))]
        chosen_int          = int(chosen)
        if chosen_int == EOT:
            pbar.close()
            break
        generated.append(chosen_int)
        pbar.set_postfix({'tokens': len(generated)})
        next_tok       = jnp.array([[chosen_int]], dtype=jnp.int32)
        logits, caches = model.apply({'params': params}, next_tok, train=False,
                                      caches=caches, cache_pos=cache_pos)
        cache_pos += 1

    return tokenizer.decode(generated)


def agentic_solve(params, query, max_retries=3):
    print(f"\nQuestion: {query}")
    print("=" * 80)

    question_type = classify_question(query)
    print(f"Classified as: {question_type.upper()}")

    attempt     = 0
    final_code  = None
    exec_result = None
    response    = ""

    for attempt in tqdm(range(1, max_retries + 1), desc="Agent attempts", unit="try"):
        print(f"\nAttempt {attempt}/{max_retries}")

        prompt   = build_prompt(query, question_type)
        response = agent_generate(params, prompt)

        print(f"\nGenerated Response:\n{response}")

        code = extract_code_block(response)

        if code:
            print(f"\nExecuting code...")
            exec_result = execute_code(code)
            print(f"Status: {exec_result['status']}")
            print(f"Output: {exec_result['output']}")

            if exec_result['status'] == 'success':
                final_code = code
                print(f"\nSuccess on attempt {attempt}")
                break
            else:
                print(f"Error found, retrying...")
        else:
            print("No code block found in response, retrying...")

    return {
        'query':       query,
        'type':        question_type,
        'response':    response,
        'final_code':  final_code,
        'exec_result': exec_result,
        'attempts':    attempt,
        'solved':      final_code is not None and exec_result is not None and exec_result['status'] == 'success'
    }

print("Agentic solver defined")

Agentic solver defined


## Agentic Solver — Results & Limitations

### What Happened
The agentic solver ran 3 attempts but **could not generate valid Python code**.  
The model produced fragments like `fromfuture. : _ = : - ` instead of real code.

### Root Cause
This model was trained with very tight sequence limits:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `MAX_SEQ_LEN` | 48 tokens | Max total sequence length |
| `GRPO_MAX_NEW` | 16 tokens | Max new tokens during GRPO training |
| Prompt cost | ~14 tokens | Tokens used by the instruction |
| **Left for code** | **~34 tokens** | Not enough for a full function |

A minimal Python function needs **~40+ tokens** — the model simply runs out of space before finishing.

### Why `generate_answer` Works But Agentic Fails
- `generate_answer` shows raw output — even fragments look okay printed directly  
- The agentic solver looks for a **complete, executable code block** — fragments always fail this check

### Fix Options

**Quick fix (no retraining):** Use `generate_answer` directly instead of the agentic loop — it shows whatever the model produces without requiring a valid code block.

**Proper fix (requires retraining Steps 4–5):**
```python
MAX_SEQ_LEN  = 256  # was 48
GRPO_MAX_NEW = 150  # was 16
```
With 256 tokens, the model has enough room to generate complete functions and the agentic solver will work as intended.

In [92]:
from tqdm.notebook import tqdm
from tabulate import tabulate

questions = [
    "is_prime function"
]

results = []

for i, question in enumerate(tqdm(questions, desc="Questions", unit="q"), 1):
    tqdm.write("\n" + "#"*80)
    tqdm.write(f"QUESTION {i}/{len(questions)}")
    tqdm.write("#"*80)
    result = agentic_solve(inference_params, question, max_retries=3)
    results.append(result)
    tqdm.write("\nFINAL RESULT")
    tqdm.write(f"Solved:   {result.get('solved', False)}")
    tqdm.write(f"Attempts: {result.get('attempts', 0)}")
    if result.get('final_code'):
        tqdm.write(f"Final Code:\n{result['final_code']}")
    if result.get('exec_result') and isinstance(result['exec_result'], dict):
        tqdm.write(f"Output:\n{result['exec_result'].get('output', '')}")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

summary = [
    [
        f"Q{i+1}",
        r.get('type', 'unknown').upper(),
        'SOLVED' if r.get('solved') else 'FAILED',
        r.get('attempts', 0)
    ]
    for i, r in enumerate(results)
]

print(tabulate(summary, headers=['Question', 'Type', 'Status', 'Attempts'], tablefmt='heavy_grid'))

Questions:   0%|          | 0/1 [00:00<?, ?q/s]


################################################################################
QUESTION 1/1
################################################################################

Question: is_prime function
Classified as: ALGORITHM


Agent attempts:   0%|          | 0/3 [00:00<?, ?try/s]


Attempt 1/3


Generating:   0%|          | 0/200 [00:00<?, ?it/s]


Generated Response:

fromfuture.
 :
   _ =
:
 
    -
            
No code block found in response, retrying...

Attempt 2/3


Generating:   0%|          | 0/200 [00:00<?, ?it/s]


Generated Response:

fromfuture.
 :
   _ =
:
 
    -
            
No code block found in response, retrying...

Attempt 3/3


Generating:   0%|          | 0/200 [00:00<?, ?it/s]


Generated Response:

fromfuture.
 :
   _ =
:
 
    -
            
No code block found in response, retrying...

FINAL RESULT
Solved:   False
Attempts: 3

SUMMARY
┏━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Question   ┃ Type      ┃ Status   ┃   Attempts ┃
┣━━━━━━━━━━━━╋━━━━━━━━━━━╋━━━━━━━━━━╋━━━━━━━━━━━━┫
┃ Q1         ┃ ALGORITHM ┃ FAILED   ┃          3 ┃
┗━━━━━━━━━━━━┻━━━━━━━━━━━┻━━━━━━━━━━┻━━━━━━━━━━━━┛
